In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, shutil, glob, re
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
EXPORT_DIR = BASE_DIR / "exports"
APP_DIR = BASE_DIR / "desktop_app"

for d in [DATA_DIR, MODEL_DIR, EXPORT_DIR, APP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Workspace created:")
print(BASE_DIR)

Mounted at /content/drive
Workspace created:
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP


In [ ]:
SEARCH_ROOT = Path("/content/drive/MyDrive")

patient_json_files = list(SEARCH_ROOT.rglob("patient_precision_TCGA*.json"))

print("Number of patient precision JSON files found:", len(patient_json_files))

for p in patient_json_files[:20]:
    print(p)

if len(patient_json_files) == 0:
    raise Exception("No patient_precision_TCGA*.json files found. Check your Drive paths.")

Number of patient precision JSON files found: 46
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_67_6217.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_91_6847.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_44_2661.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_05_4402.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_64_1681.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_69_7765.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_50_6595.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_49_4494.json
/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_38_6178.json
/content/drive/

In [ ]:
def safe_get(d, keys, default=None):
    if not isinstance(d, dict):
        return default

    for key in keys:
        if key in d and d[key] not in [None, "", "Unknown", "unknown", "NA", "N/A"]:
            return d[key]

    return default


def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        if x.strip() == "":
            return []
        # Split only if it looks like a list
        if "," in x:
            return [i.strip() for i in x.split(",") if i.strip()]
        return [x]
    return [x]


def normalize_number(x, default=np.nan):
    try:
        if x in [None, "", "Unknown", "unknown", "NA", "N/A"]:
            return default
        return float(x)
    except:
        return default


def normalize_stage(stage):
    s = str(stage).upper()

    if "IV" in s or "STAGE IV" in s or "M1" in s:
        return "IV"
    if "III" in s or "STAGE III" in s:
        return "III"
    if "II" in s or "STAGE II" in s:
        return "II"
    if "I" in s or "STAGE I" in s:
        return "I"

    return "Unknown"


def normalize_egfr_class(x):
    s = str(x).lower()

    if "exon 19" in s or "19 deletion" in s or "elrea" in s or "del" in s:
        return "EGFR exon 19 deletion"
    if "l858r" in s:
        return "EGFR L858R"
    if "t790m" in s:
        return "EGFR T790M resistance"
    if "exon 20" in s or "ins" in s:
        return "EGFR exon 20 insertion"
    if "negative" in s or "wild" in s:
        return "EGFR negative"
    if "egfr" in s:
        return "Other EGFR mutation"

    return "Unknown EGFR"


def get_therapy_list(raw):
    therapy_keys = [
        "ranked_therapies",
        "treatments",
        "therapy_ranking",
        "recommended_treatments",
        "ranked_treatments",
        "final_ranked_drugs",
        "drug_ranking"
    ]

    therapies = safe_get(raw, therapy_keys, [])

    if isinstance(therapies, dict):
        therapies = list(therapies.values())

    if not isinstance(therapies, list):
        therapies = []

    clean = []

    for i, t in enumerate(therapies):
        if isinstance(t, dict):
            drug = safe_get(t, ["drug", "name", "treatment", "therapy"], "Unknown")
            eff = normalize_number(safe_get(t, ["eff", "efficiency", "predicted_efficiency", "score"], 0.7), 0.7)
            res = normalize_number(safe_get(t, ["res", "resistance", "resistance_risk"], 0.4), 0.4)

            if eff > 1:
                eff = eff / 100
            if res > 1:
                res = res / 100

            clean.append({
                "rank": int(safe_get(t, ["rank"], i + 1)),
                "drug": drug,
                "efficiency": float(np.clip(eff, 0, 1)),
                "resistance": float(np.clip(res, 0, 1))
            })

        elif isinstance(t, str):
            clean.append({
                "rank": i + 1,
                "drug": t,
                "efficiency": max(0.35, 0.85 - i * 0.05),
                "resistance": min(0.85, 0.25 + i * 0.06)
            })

    return clean


DEFAULT_DRUG_LIBRARY = [
    "Osimertinib",
    "Lazertinib",
    "Amivantamab",
    "Amivantamab + Lazertinib",
    "Osimertinib + Chemotherapy",
    "Afatinib",
    "Dacomitinib",
    "Gefitinib",
    "Erlotinib",
    "Platinum-based Chemotherapy"
]


def default_therapies_for_patient(egfr_class, has_met, has_tp53):
    therapies = []

    for i, drug in enumerate(DEFAULT_DRUG_LIBRARY):
        base_eff = 0.55
        base_res = 0.45

        if egfr_class in ["EGFR exon 19 deletion", "EGFR L858R"]:
            if drug == "Osimertinib":
                base_eff = 0.86
                base_res = 0.34
            elif drug == "Osimertinib + Chemotherapy":
                base_eff = 0.88
                base_res = 0.31
            elif drug == "Amivantamab + Lazertinib":
                base_eff = 0.90 if has_met else 0.82
                base_res = 0.25 if has_met else 0.36
            elif drug in ["Afatinib", "Dacomitinib"]:
                base_eff = 0.74
                base_res = 0.48
            elif drug in ["Gefitinib", "Erlotinib"]:
                base_eff = 0.68
                base_res = 0.55
            elif drug == "Platinum-based Chemotherapy":
                base_eff = 0.61
                base_res = 0.52

        elif egfr_class == "EGFR T790M resistance":
            if drug == "Osimertinib":
                base_eff = 0.82
                base_res = 0.42
            elif drug == "Amivantamab + Lazertinib":
                base_eff = 0.78
                base_res = 0.38
            elif drug == "Platinum-based Chemotherapy":
                base_eff = 0.58
                base_res = 0.57

        elif egfr_class == "EGFR exon 20 insertion":
            if "Amivantamab" in drug:
                base_eff = 0.74
                base_res = 0.44
            elif drug == "Platinum-based Chemotherapy":
                base_eff = 0.60
                base_res = 0.55
            else:
                base_eff = 0.45
                base_res = 0.65

        elif egfr_class == "EGFR negative":
            if drug == "Platinum-based Chemotherapy":
                base_eff = 0.62
                base_res = 0.55
            else:
                base_eff = 0.35
                base_res = 0.70

        if has_tp53:
            base_res += 0.08
            base_eff -= 0.03

        therapies.append({
            "rank": i + 1,
            "drug": drug,
            "efficiency": float(np.clip(base_eff, 0.05, 1.0)),
            "resistance": float(np.clip(base_res, 0.0, 1.0))
        })

    therapies = sorted(therapies, key=lambda x: (-x["efficiency"], x["resistance"]))

    for i, t in enumerate(therapies):
        t["rank"] = i + 1

    return therapies


patient_rows = []
therapy_rows = []
raw_patients = {}

for fp in patient_json_files:
    with open(fp, "r", encoding="utf-8") as f:
        raw = json.load(f)

    pid = safe_get(raw, ["patient_id", "id", "case_id"], fp.stem.replace("patient_precision_", ""))

    age = normalize_number(safe_get(raw, ["age", "Age", "age_at_diagnosis", "Age at diagnosis"], np.nan))
    sex = safe_get(raw, ["sex", "gender", "Sex", "Gender"], "Unknown")
    stage = normalize_stage(safe_get(raw, ["stage", "tumor_stage", "Tumor stage", "ajcc_pathologic_stage", "ajcc_pathologic_t"], "Unknown"))
    survival_status = safe_get(raw, ["survival_status", "vital_status", "status"], "Unknown")

    egfr_raw = safe_get(raw, [
        "primary_egfr", "raw_egfr", "raw_egfr_mutation",
        "egfr_status", "egfr", "EGFR", "EGFR_mutation", "mutation"
    ], "Unknown")

    egfr_class = normalize_egfr_class(egfr_raw)

    secondary = normalize_list(safe_get(raw, [
        "secondary_mutations",
        "secondary_alterations",
        "co_mutations",
        "alterations"
    ], []))

    resistance_mechanisms = normalize_list(safe_get(raw, [
        "resistance_mechanisms",
        "resistance"
    ], []))

    has_tp53 = int(any("TP53" in str(x).upper() for x in secondary))
    has_met = int(any("MET" in str(x).upper() for x in secondary))
    has_kras = int(any("KRAS" in str(x).upper() for x in secondary))
    has_pik3ca = int(any("PIK3CA" in str(x).upper() for x in secondary))
    has_bypass = int(any("bypass" in str(x).lower() for x in resistance_mechanisms))
    has_resistance = int(len(resistance_mechanisms) > 0)

    # Optional scores if already exist in your JSON
    egfr_pathway_activity = normalize_number(safe_get(raw, [
        "egfr_pathway_activity", "EGFR_pathway_activity", "pathway_activity"
    ], np.nan))

    tumor_aggressiveness = normalize_number(safe_get(raw, [
        "tumor_aggressiveness", "aggressiveness_score", "tumor_aggressiveness_score"
    ], np.nan))

    pathology_score = normalize_number(safe_get(raw, [
        "pathology_score", "pathology_risk", "qupath_score", "cellularity_score"
    ], np.nan))

    immune_score = normalize_number(safe_get(raw, [
        "immune_score", "immune_activity", "immune_microenvironment_score"
    ], np.nan))

    therapies = get_therapy_list(raw)

    if len(therapies) == 0:
        therapies = default_therapies_for_patient(egfr_class, has_met, has_tp53)

    best = sorted(therapies, key=lambda x: x["rank"])[0]

    patient_record = {
        "patient_id": pid,
        "age": age,
        "sex": sex,
        "stage": stage,
        "survival_status": survival_status,
        "egfr_raw": egfr_raw,
        "egfr_class": egfr_class,
        "secondary_mutations": secondary,
        "resistance_mechanisms": resistance_mechanisms,
        "has_TP53": has_tp53,
        "has_MET": has_met,
        "has_KRAS": has_kras,
        "has_PIK3CA": has_pik3ca,
        "has_bypass": has_bypass,
        "has_resistance_mechanism": has_resistance,
        "egfr_pathway_activity": egfr_pathway_activity,
        "tumor_aggressiveness": tumor_aggressiveness,
        "pathology_score": pathology_score,
        "immune_score": immune_score,
        "best_known_treatment": best["drug"],
        "best_known_efficiency": best["efficiency"],
        "best_known_resistance": best["resistance"]
    }

    patient_rows.append(patient_record)

    for t in therapies:
        therapy_rows.append({
            **patient_record,
            "drug": t["drug"],
            "rank": t["rank"],
            "target_efficiency": t["efficiency"],
            "target_resistance": t["resistance"],
            "is_best_treatment": int(t["rank"] == 1)
        })

    raw_patients[pid] = raw


patients_df = pd.DataFrame(patient_rows)
therapy_df = pd.DataFrame(therapy_rows)

# Fill numerical missing values
for col in ["age", "egfr_pathway_activity", "tumor_aggressiveness", "pathology_score", "immune_score"]:
    if col in patients_df.columns:
        median = patients_df[col].median() if patients_df[col].notna().any() else 0.5
        patients_df[col] = patients_df[col].fillna(median)
        therapy_df[col] = therapy_df[col].fillna(median)

patients_df.to_csv(DATA_DIR / "patients_model_table.csv", index=False)
therapy_df.to_csv(DATA_DIR / "patient_treatment_training_table.csv", index=False)

print("Patients table:", patients_df.shape)
print("Patient-treatment table:", therapy_df.shape)

display(patients_df.head())
display(therapy_df.head())

Patients table: (46, 22)
Patient-treatment table: (460, 27)


,patient_id,age,sex,stage,survival_status,egfr_raw,egfr_class,secondary_mutations,resistance_mechanisms,has_TP53,...,has_PIK3CA,has_bypass,has_resistance_mechanism,egfr_pathway_activity,tumor_aggressiveness,pathology_score,immune_score,best_known_treatment,best_known_efficiency,best_known_resistance
0,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
1,TCGA-91-6847,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
2,TCGA-44-2661,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
3,TCGA-05-4402,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45
4,TCGA-64-1681,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0,0,0,0.5,0.5,0.5,0.5,Osimertinib,0.55,0.45


,patient_id,age,sex,stage,survival_status,egfr_raw,egfr_class,secondary_mutations,resistance_mechanisms,has_TP53,...,pathology_score,immune_score,best_known_treatment,best_known_efficiency,best_known_resistance,drug,rank,target_efficiency,target_resistance,is_best_treatment
0,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Osimertinib,1,0.55,0.45,1
1,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Lazertinib,2,0.55,0.45,0
2,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Amivantamab,3,0.55,0.45,0
3,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Amivantamab + Lazertinib,4,0.55,0.45,0
4,TCGA-67-6217,0.5,Unknown,Unknown,Unknown,Unknown,Unknown EGFR,[],[],0,...,0.5,0.5,Osimertinib,0.55,0.45,Osimertinib + Chemotherapy,5,0.55,0.45,0


In [ ]:
# Cellule vidée pour corriger l'erreur de syntaxe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, re
import pandas as pd
import numpy as np

SEARCH_ROOT = Path("/content/drive/MyDrive")

patient_json_files = list(SEARCH_ROOT.rglob("patient_precision_TCGA*.json"))

print("Found:", len(patient_json_files))

for i, p in enumerate(patient_json_files[:30]):
    print(i, p)

# Inspect one important patient
target_files = [
    p for p in patient_json_files
    if "44" in p.name or "2661" in p.name or "05" in p.name or "4402" in p.name
]

fp = target_files[0] if target_files else patient_json_files[0]

print("\nInspecting file:")
print(fp)

with open(fp, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("\nTop-level keys:")
print(list(raw.keys()))

print("\nFirst 5000 characters:")
print(json.dumps(raw, indent=2, ensure_ascii=False)[:5000])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: 46
0 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_67_6217.json
1 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_91_6847.json
2 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_44_2661.json
3 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_05_4402.json
4 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_64_1681.json
5 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_69_7765.json
6 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_50_6595.json
7 /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_49_4494.json
8 /content/dr

In [ ]:
import json, re, os
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOT = Path("/content/drive/MyDrive")

# Prefer the known precision folder if it exists
preferred_dirs = [
    Path("/content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision"),
    Path("/content/drive/MyDrive/pfe_digital_twin_app/data/patient_precision"),
    Path("/content/drive/MyDrive/pfe_digital_twin/Data/patient_precision"),
    Path("/content/drive/MyDrive/pfe_digital_twin/data/patient_precision"),
]

patient_json_files = []

for d in preferred_dirs:
    if d.exists():
        patient_json_files = list(d.glob("patient_precision_TCGA*.json"))
        print("Using preferred folder:", d)
        break

if len(patient_json_files) == 0:
    print("Preferred folder not found. Searching whole Drive...")
    patient_json_files = list(SEARCH_ROOT.rglob("patient_precision_TCGA*.json"))

print("Raw JSON files found:", len(patient_json_files))

# Deduplicate by patient ID
def patient_id_from_filename(path):
    name = path.stem.replace("patient_precision_", "")
    name = name.replace("_", "-")
    name = re.sub(r"TCGA-(\d+)-(\d+).*", r"TCGA-\1-\2", name)
    return name

dedup = {}
for p in patient_json_files:
    pid = patient_id_from_filename(p)
    # keep first occurrence
    if pid not in dedup:
        dedup[pid] = p

patient_json_files = list(dedup.values())

print("Unique patient JSON files:", len(patient_json_files))
for p in patient_json_files[:10]:
    print("-", p)


def flatten_json(obj, prefix=""):
    items = {}

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_key = f"{prefix}.{k}" if prefix else str(k)
            items.update(flatten_json(v, new_key))

    elif isinstance(obj, list):
        # keep full list
        items[prefix] = obj

        # also flatten first few elements if dict
        for i, v in enumerate(obj[:5]):
            new_key = f"{prefix}[{i}]"
            items.update(flatten_json(v, new_key))

    else:
        items[prefix] = obj

    return items


def clean_value(v):
    if v is None:
        return None
    if isinstance(v, str):
        if v.strip() in ["", "Unknown", "unknown", "NA", "N/A", "nan"]:
            return None
    return v


def find_by_key_contains(flat, include_terms, exclude_terms=None):
    exclude_terms = exclude_terms or []

    include_terms = [t.lower() for t in include_terms]
    exclude_terms = [t.lower() for t in exclude_terms]

    candidates = []

    for k, v in flat.items():
        kl = k.lower()
        v_clean = clean_value(v)

        if v_clean is None:
            continue

        if all(t in kl for t in include_terms) and not any(e in kl for e in exclude_terms):
            candidates.append((k, v_clean))

    if candidates:
        # prefer shorter key path
        candidates = sorted(candidates, key=lambda x: len(x[0]))
        return candidates[0][1], candidates[0][0]

    return None, None


def find_first_existing(flat, key_options):
    for opt in key_options:
        opt_l = opt.lower()

        # exact ending match
        for k, v in flat.items():
            if k.lower().endswith(opt_l):
                v_clean = clean_value(v)
                if v_clean is not None:
                    return v_clean, k

        # contains match
        for k, v in flat.items():
            if opt_l in k.lower():
                v_clean = clean_value(v)
                if v_clean is not None:
                    return v_clean, k

    return None, None


def as_list(x):
    if x is None:
        return []

    if isinstance(x, list):
        out = []
        for item in x:
            if isinstance(item, dict):
                out.append(json.dumps(item, ensure_ascii=False))
            else:
                out.append(str(item))
        return out

    if isinstance(x, str):
        if x.strip() == "":
            return []
        if "," in x:
            return [i.strip() for i in x.split(",") if i.strip()]
        return [x]

    return [str(x)]


def to_float(x, default=np.nan):
    try:
        if x is None:
            return default
        return float(x)
    except:
        return default


def normalize_stage(x):
    if x is None:
        return "Unknown"

    s = str(x).upper()

    if "STAGE IV" in s or s == "IV" or "M1" in s:
        return "IV"
    if "STAGE III" in s or s == "III":
        return "III"
    if "STAGE II" in s or s == "II":
        return "II"
    if "STAGE I" in s or s == "I":
        return "I"

    if "T4" in s or "T3" in s:
        return "III/IV"
    if "T2" in s:
        return "II"
    if "T1" in s:
        return "I"

    return str(x)


def normalize_egfr(x):
    if x is None:
        return "Unknown EGFR"

    s = str(x).lower()

    if "exon 19" in s or "19 deletion" in s or "elrea" in s or "del" in s:
        return "EGFR exon 19 deletion"
    if "l858r" in s:
        return "EGFR L858R"
    if "t790m" in s:
        return "EGFR T790M resistance"
    if "exon 20" in s or "ins" in s:
        return "EGFR exon 20 insertion"
    if "negative" in s or "wild" in s:
        return "EGFR negative"
    if "egfr" in s or "p." in s:
        return "Other EGFR mutation"

    return "Unknown EGFR"


def extract_therapy_list(raw, flat, egfr_class, has_met, has_tp53):
    possible_keys = [
        "ranked_therapies",
        "treatments",
        "therapy_ranking",
        "recommended_treatments",
        "ranked_treatments",
        "final_ranked_drugs",
        "drug_ranking",
        "recommended_drugs"
    ]

    therapies = None
    source_key = None

    for k, v in flat.items():
        kl = k.lower()
        if any(pk in kl for pk in possible_keys):
            if isinstance(v, list):
                therapies = v
                source_key = k
                break

    if therapies is None:
        # Try top-level/nested normal search
        for k in possible_keys:
            v, found_key = find_first_existing(flat, [k])
            if isinstance(v, list):
                therapies = v
                source_key = found_key
                break

    clean = []

    if isinstance(therapies, list):
        for i, t in enumerate(therapies):
            if isinstance(t, dict):
                tf = flatten_json(t)

                drug, _ = find_first_existing(tf, ["drug", "name", "treatment", "therapy"])
                eff, _ = find_first_existing(tf, ["efficiency", "eff", "predicted_efficiency", "score"])
                res, _ = find_first_existing(tf, ["resistance", "res", "resistance_risk"])

                drug = drug or f"Treatment {i+1}"
                eff = to_float(eff, 0.7)
                res = to_float(res, 0.4)

                if eff > 1:
                    eff = eff / 100
                if res > 1:
                    res = res / 100

                clean.append({
                    "rank": i + 1,
                    "drug": str(drug),
                    "efficiency": float(np.clip(eff, 0, 1)),
                    "resistance": float(np.clip(res, 0, 1))
                })

            elif isinstance(t, str):
                clean.append({
                    "rank": i + 1,
                    "drug": t,
                    "efficiency": max(0.40, 0.85 - i * 0.05),
                    "resistance": min(0.90, 0.30 + i * 0.06)
                })

    if len(clean) > 0:
        return clean, source_key

    return default_therapies_for_patient(egfr_class, has_met, has_tp53), "default_generated"


DEFAULT_DRUG_LIBRARY = [
    "Osimertinib",
    "Lazertinib",
    "Amivantamab",
    "Amivantamab + Lazertinib",
    "Osimertinib + Chemotherapy",
    "Afatinib",
    "Dacomitinib",
    "Gefitinib",
    "Erlotinib",
    "Platinum-based Chemotherapy"
]


def default_therapies_for_patient(egfr_class, has_met, has_tp53):
    therapies = []

    for drug in DEFAULT_DRUG_LIBRARY:
        eff = 0.55
        res = 0.45

        if egfr_class in ["EGFR exon 19 deletion", "EGFR L858R"]:
            if drug == "Amivantamab + Lazertinib":
                eff = 0.92 if has_met else 0.84
                res = 0.24 if has_met else 0.36
            elif drug == "Osimertinib + Chemotherapy":
                eff = 0.88
                res = 0.32
            elif drug == "Osimertinib":
                eff = 0.86
                res = 0.34
            elif drug in ["Afatinib", "Dacomitinib"]:
                eff = 0.74
                res = 0.48
            elif drug in ["Gefitinib", "Erlotinib"]:
                eff = 0.68
                res = 0.56
            elif drug == "Platinum-based Chemotherapy":
                eff = 0.61
                res = 0.53

        elif egfr_class == "EGFR T790M resistance":
            if drug == "Osimertinib":
                eff = 0.82
                res = 0.42
            elif drug == "Amivantamab + Lazertinib":
                eff = 0.78
                res = 0.38
            elif drug == "Platinum-based Chemotherapy":
                eff = 0.58
                res = 0.57

        elif egfr_class == "EGFR exon 20 insertion":
            if "Amivantamab" in drug:
                eff = 0.76
                res = 0.44
            elif drug == "Platinum-based Chemotherapy":
                eff = 0.60
                res = 0.55
            else:
                eff = 0.45
                res = 0.65

        elif egfr_class == "EGFR negative":
            if drug == "Platinum-based Chemotherapy":
                eff = 0.62
                res = 0.55
            else:
                eff = 0.35
                res = 0.70

        if has_tp53:
            eff -= 0.03
            res += 0.08

        therapies.append({
            "rank": 0,
            "drug": drug,
            "efficiency": float(np.clip(eff, 0.05, 1)),
            "resistance": float(np.clip(res, 0, 1))
        })

    therapies = sorted(therapies, key=lambda x: (-x["efficiency"], x["resistance"]))

    for i, t in enumerate(therapies):
        t["rank"] = i + 1

    return therapies


patient_rows = []
therapy_rows = []
debug_rows = []

for fp in patient_json_files:
    with open(fp, "r", encoding="utf-8") as f:
        raw = json.load(f)

    flat = flatten_json(raw)

    pid = patient_id_from_filename(fp)

    patient_id_val, patient_id_key = find_first_existing(flat, ["patient_id", "case_id", "submitter_id", "barcode"])
    if patient_id_val:
        pid = str(patient_id_val)

    age, age_key = find_first_existing(flat, ["age_at_diagnosis", "age at diagnosis", "age"])
    sex, sex_key = find_first_existing(flat, ["gender", "sex"])
    stage, stage_key = find_first_existing(flat, ["tumor_stage", "tumor stage", "stage", "ajcc_pathologic_stage", "ajcc_pathologic_t"])
    survival, survival_key = find_first_existing(flat, ["survival_status", "vital_status", "status"])

    # EGFR: prefer explicit EGFR mutation/profile keys
    egfr, egfr_key = find_first_existing(flat, [
        "primary_egfr",
        "raw_egfr",
        "raw_egfr_mutation",
        "egfr_status",
        "egfr_class",
        "EGFR_mutation",
        "egfr mutation",
        "egfr"
    ])

    # If no key found, search values containing EGFR/p./exon
    if egfr is None:
        for k, v in flat.items():
            if isinstance(v, str):
                vl = v.lower()
                if "egfr" in vl or "exon 19" in vl or "l858r" in vl or "t790m" in vl or "p." in vl:
                    egfr = v
                    egfr_key = k
                    break

    egfr_class = normalize_egfr(egfr)

    secondary, sec_key = find_first_existing(flat, [
        "secondary_mutations",
        "secondary_alterations",
        "co_mutations",
        "alterations",
        "mutations"
    ])

    resistance, resistance_key = find_first_existing(flat, [
        "resistance_mechanisms",
        "resistance mechanisms",
        "resistance_notes",
        "resistance"
    ])

    secondary_list = as_list(secondary)
    resistance_list = as_list(resistance)

    # Try to detect common secondary mutations from full flattened content
    all_text = json.dumps(raw, ensure_ascii=False).upper()

    has_tp53 = int("TP53" in all_text)
    has_met = int(re.search(r"\bMET\b", all_text) is not None)
    has_kras = int("KRAS" in all_text)
    has_pik3ca = int("PIK3CA" in all_text)
    has_bypass = int("BYPASS" in all_text or "MET BYPASS" in all_text)
    has_resistance = int(len(resistance_list) > 0 or "RESISTANCE" in all_text)

    egfr_pathway, egfr_pathway_key = find_first_existing(flat, [
        "egfr_pathway_activity",
        "pathway_activity",
        "egfr activity"
    ])

    tumor_aggr, tumor_aggr_key = find_first_existing(flat, [
        "tumor_aggressiveness",
        "aggressiveness_score",
        "aggressiveness"
    ])

    pathology_score, pathology_key = find_first_existing(flat, [
        "pathology_score",
        "pathology_risk",
        "qupath_score",
        "cellularity_score"
    ])

    immune_score, immune_key = find_first_existing(flat, [
        "immune_score",
        "immune_activity"
    ])

    therapies, therapy_source = extract_therapy_list(raw, flat, egfr_class, has_met, has_tp53)
    best = sorted(therapies, key=lambda x: x["rank"])[0]

    record = {
        "patient_id": pid,
        "age": to_float(age, np.nan),
        "sex": sex if sex is not None else "Unknown",
        "stage": normalize_stage(stage),
        "survival_status": survival if survival is not None else "Unknown",
        "egfr_raw": egfr if egfr is not None else "Unknown",
        "egfr_class": egfr_class,
        "secondary_mutations": secondary_list,
        "resistance_mechanisms": resistance_list,
        "has_TP53": has_tp53,
        "has_MET": has_met,
        "has_KRAS": has_kras,
        "has_PIK3CA": has_pik3ca,
        "has_bypass": has_bypass,
        "has_resistance_mechanism": has_resistance,
        "egfr_pathway_activity": to_float(egfr_pathway, np.nan),
        "tumor_aggressiveness": to_float(tumor_aggr, np.nan),
        "pathology_score": to_float(pathology_score, np.nan),
        "immune_score": to_float(immune_score, np.nan),
        "best_known_treatment": best["drug"],
        "best_known_efficiency": best["efficiency"],
        "best_known_resistance": best["resistance"]
    }

    patient_rows.append(record)

    for t in therapies:
        therapy_rows.append({
            **record,
            "drug": t["drug"],
            "rank": t["rank"],
            "target_efficiency": t["efficiency"],
            "target_resistance": t["resistance"],
            "is_best_treatment": int(t["rank"] == 1)
        })

    debug_rows.append({
        "patient_id": pid,
        "file": str(fp),
        "age_key": age_key,
        "sex_key": sex_key,
        "stage_key": stage_key,
        "survival_key": survival_key,
        "egfr_key": egfr_key,
        "secondary_key": sec_key,
        "resistance_key": resistance_key,
        "therapy_source": therapy_source
    })


patients_df = pd.DataFrame(patient_rows)
therapy_df = pd.DataFrame(therapy_rows)
debug_df = pd.DataFrame(debug_rows)

# Fill missing numeric features, but keep real clinical missing visible
for col in ["egfr_pathway_activity", "tumor_aggressiveness", "pathology_score", "immune_score"]:
    if patients_df[col].notna().any():
        fill = patients_df[col].median()
    else:
        fill = 0.5

    patients_df[col] = patients_df[col].fillna(fill)
    therapy_df[col] = therapy_df[col].fillna(fill)

# If age is normalized 0-1, keep it; if missing, use median
if patients_df["age"].notna().any():
    age_fill = patients_df["age"].median()
else:
    age_fill = 60

patients_df["age"] = patients_df["age"].fillna(age_fill)
therapy_df["age"] = therapy_df["age"].fillna(age_fill)

patients_df.to_csv(DATA_DIR / "patients_model_table_fixed.csv", index=False)
therapy_df.to_csv(DATA_DIR / "patient_treatment_training_table_fixed.csv", index=False)
debug_df.to_csv(DATA_DIR / "extraction_debug_keys.csv", index=False)

print("Patients table fixed:", patients_df.shape)
print("Therapy table fixed:", therapy_df.shape)

display(patients_df.head(10))
display(debug_df.head(10))

print("\nUnknown counts:")
print((patients_df[["sex", "stage", "survival_status", "egfr_class"]] == "Unknown").sum())
print("Unknown EGFR count:", (patients_df["egfr_class"] == "Unknown EGFR").sum())

Using preferred folder: /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision
Preferred folder not found. Searching whole Drive...
Raw JSON files found: 46
Unique patient JSON files: 23
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_67_6217.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_91_6847.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_44_2661.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_05_4402.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_64_1681.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_69_7765.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_50_6595.json
- /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_

,patient_id,age,sex,stage,survival_status,egfr_raw,egfr_class,secondary_mutations,resistance_mechanisms,has_TP53,...,has_PIK3CA,has_bypass,has_resistance_mechanism,egfr_pathway_activity,tumor_aggressiveness,pathology_score,immune_score,best_known_treatment,best_known_efficiency,best_known_resistance
0,TCGA-67-6217,0.714286,FEMALE,II,LIVING,EGFR L858R,EGFR L858R,"[STK11, KEAP1, NRAS]",[Metabolic/stress-resistance phenotype],0,...,0,1,1,0.351536,0.391193,0.5,0.5,Amivantamab + Lazertinib,0.92,0.24
1,TCGA-91-6847,0.321429,FEMALE,I,LIVING,Uncommon sensitizing EGFR mutation,Other EGFR mutation,"[TP53, KRAS, MET, ERBB2, BRAF, RB1, NRAS, AKT1]","[MET bypass signaling, PI3K/AKT pathway bypass...",1,...,0,1,1,0.131961,0.307413,0.5,0.5,Osimertinib,0.52,0.53
2,TCGA-44-2661,0.571429,FEMALE,I,LIVING,EGFR exon 19 deletion,EGFR exon 19 deletion,"[TP53, MET]","[MET bypass signaling, Genomic instability / a...",1,...,0,1,1,0.391296,0.355233,0.5,0.5,Amivantamab + Lazertinib,0.89,0.32
3,TCGA-05-4402,0.142857,FEMALE,IV,DECEASED,EGFR exon 19 deletion,EGFR exon 19 deletion,"[TP53, KRAS, PIK3CA, MET, ERBB2, BRAF, NF1, NR...","[MET bypass signaling, PI3K/AKT pathway bypass...",1,...,1,1,1,0.439866,0.462614,0.5,0.5,Amivantamab + Lazertinib,0.89,0.32
4,TCGA-64-1681,0.285714,FEMALE,I,DECEASED,EGFR L858R,EGFR L858R,"[TP53, KRAS, MET, BRAF, NRAS]","[MET bypass signaling, MAPK pathway bypass, Ge...",1,...,0,1,1,0.374597,0.234042,0.5,0.5,Amivantamab + Lazertinib,0.89,0.32
5,TCGA-69-7765,0.107143,MALE,IV,LIVING,Rare/uncertain EGFR mutation,Other EGFR mutation,"[TP53, MET, BRAF, NF1, AKT1]","[MET bypass signaling, PI3K/AKT pathway bypass...",1,...,0,1,1,0.486661,0.525138,0.5,0.5,Osimertinib,0.52,0.53
6,TCGA-50-6595,0.750000,FEMALE,III,DECEASED,Uncommon sensitizing EGFR mutation,Other EGFR mutation,"[TP53, PIK3CA]","[PI3K/AKT pathway bypass, Genomic instability ...",1,...,1,1,1,0.133494,0.426230,0.5,0.5,Osimertinib,0.52,0.53
7,TCGA-49-4494,0.857143,MALE,III,DECEASED,Compound EGFR mutation with T790M resistance,EGFR T790M resistance,"[MET, BRAF]","[EGFR T790M-mediated resistance, MET bypass si...",0,...,0,1,1,0.320582,0.474721,0.5,0.5,Osimertinib,0.82,0.42
8,TCGA-38-6178,0.607143,FEMALE,III,LIVING,EGFR exon 19 deletion,EGFR exon 19 deletion,"[TP53, MET, BRAF, NRAS]","[MET bypass signaling, MAPK pathway bypass, Ge...",1,...,0,1,1,0.211500,0.484778,0.5,0.5,Amivantamab + Lazertinib,0.89,0.32
9,TCGA-38-4627,0.392857,FEMALE,II,DECEASED,EGFR L858R,EGFR L858R,"[MET, BRAF, AKT1]","[MET bypass signaling, PI3K/AKT pathway bypass...",0,...,0,1,1,0.390601,0.457603,0.5,0.5,Amivantamab + Lazertinib,0.92,0.24


,patient_id,file,age_key,sex_key,stage_key,survival_key,egfr_key,secondary_key,resistance_key,therapy_source
0,TCGA-67-6217,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
1,TCGA-91-6847,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
2,TCGA-44-2661,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
3,TCGA-05-4402,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
4,TCGA-64-1681,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
5,TCGA-69-7765,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
6,TCGA-50-6595,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
7,TCGA-49-4494,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
8,TCGA-38-6178,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
9,TCGA-38-4627,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated



Unknown counts:
sex                0
stage              0
survival_status    0
egfr_class         0
dtype: int64
Unknown EGFR count: 0


In [ ]:
display(patients_df[["patient_id", "age", "sex", "stage", "egfr_raw", "egfr_class", "has_TP53", "has_MET", "best_known_treatment"]].head(20))

,patient_id,age,sex,stage,egfr_raw,egfr_class,has_TP53,has_MET,best_known_treatment
0,TCGA-67-6217,0.714286,FEMALE,II,EGFR L858R,EGFR L858R,0,1,Amivantamab + Lazertinib
1,TCGA-91-6847,0.321429,FEMALE,I,Uncommon sensitizing EGFR mutation,Other EGFR mutation,1,1,Osimertinib
2,TCGA-44-2661,0.571429,FEMALE,I,EGFR exon 19 deletion,EGFR exon 19 deletion,1,1,Amivantamab + Lazertinib
3,TCGA-05-4402,0.142857,FEMALE,IV,EGFR exon 19 deletion,EGFR exon 19 deletion,1,1,Amivantamab + Lazertinib
4,TCGA-64-1681,0.285714,FEMALE,I,EGFR L858R,EGFR L858R,1,1,Amivantamab + Lazertinib
5,TCGA-69-7765,0.107143,MALE,IV,Rare/uncertain EGFR mutation,Other EGFR mutation,1,1,Osimertinib
6,TCGA-50-6595,0.750000,FEMALE,III,Uncommon sensitizing EGFR mutation,Other EGFR mutation,1,1,Osimertinib
7,TCGA-49-4494,0.857143,MALE,III,Compound EGFR mutation with T790M resistance,EGFR T790M resistance,0,1,Osimertinib
8,TCGA-38-6178,0.607143,FEMALE,III,EGFR exon 19 deletion,EGFR exon 19 deletion,1,1,Amivantamab + Lazertinib
9,TCGA-38-4627,0.392857,FEMALE,II,EGFR L858R,EGFR L858R,0,1,Amivantamab + Lazertinib


In [ ]:
# Show debug keys used for TCGA-44-2661 and TCGA-05-4402
debug_df[
    debug_df["patient_id"].astype(str).str.contains("44|2661|05|4402", case=False, na=False)
]

,patient_id,file,age_key,sex_key,stage_key,survival_key,egfr_key,secondary_key,resistance_key,therapy_source
2,TCGA-44-2661,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
3,TCGA-05-4402,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
7,TCGA-49-4494,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
13,TCGA-44-6147,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated
20,TCGA-05-4410,/content/drive/MyDrive/pfe_digital_twin_app/Da...,clinical.age,clinical.sex,clinical.tumor_stage,clinical.survival_status,molecular.primary_egfr_mutation,molecular.secondary_mutations,molecular.resistance_mechanisms,default_generated


In [ ]:
# Print the flattened keys of one problematic patient
problem = patient_json_files[0]

for p in patient_json_files:
    if "44" in p.name or "2661" in p.name:
        problem = p
        break

print("Problem file:", problem)

with open(problem, "r", encoding="utf-8") as f:
    raw_problem = json.load(f)

flat_problem = flatten_json(raw_problem)

for k, v in list(flat_problem.items())[:200]:
    print(k, "=", str(v)[:150])

Problem file: /content/drive/MyDrive/pfe_digital_twin_app/Data/patient_precision /patient_precision_TCGA_44_2661.json
patient_id = TCGA-44-2661
clinical.age = 0.5714285714285712
clinical.sex = FEMALE
clinical.tumor_stage = Stage IA
clinical.survival_status = LIVING
molecular.primary_egfr_mutation = EGFR exon 19 deletion
molecular.raw_egfr_mutation = p.ELREA746del
molecular.egfr_variant_classes = ['Exon 19 deletion']
molecular.egfr_variant_classes[0] = Exon 19 deletion
molecular.secondary_mutations = ['TP53', 'MET']
molecular.secondary_mutations[0] = TP53
molecular.secondary_mutations[1] = MET
molecular.resistance_mechanisms = ['MET bypass signaling', 'Genomic instability / aggressive phenotype']
molecular.resistance_mechanisms[0] = MET bypass signaling
molecular.resistance_mechanisms[1] = Genomic instability / aggressive phenotype
base_scores.egfr_pathway_activity = 0.3912961051101855
base_scores.tumor_aggressiveness = 0.35523289345043607
base_scores.survival_risk_score = 0.27749523554

In [ ]:
print("Patients:", patients_df.shape)
print("Treatments:", therapy_df.shape)

display(
    patients_df[
        [
            "patient_id",
            "age",
            "sex",
            "stage",
            "survival_status",
            "egfr_raw",
            "egfr_class",
            "secondary_mutations",
            "resistance_mechanisms",
            "has_TP53",
            "has_MET",
            "best_known_treatment",
            "best_known_efficiency",
            "best_known_resistance"
        ]
    ].head(20)
)

print("\nUnknown EGFR count:")
print((patients_df["egfr_class"] == "Unknown EGFR").sum(), "/", len(patients_df))

print("\nUnknown sex count:")
print((patients_df["sex"] == "Unknown").sum(), "/", len(patients_df))

print("\nUnknown stage count:")
print((patients_df["stage"] == "Unknown").sum(), "/", len(patients_df))

Patients: (23, 22)
Treatments: (230, 27)


,patient_id,age,sex,stage,survival_status,egfr_raw,egfr_class,secondary_mutations,resistance_mechanisms,has_TP53,has_MET,best_known_treatment,best_known_efficiency,best_known_resistance
0,TCGA-67-6217,0.714286,FEMALE,II,LIVING,EGFR L858R,EGFR L858R,"[STK11, KEAP1, NRAS]",[Metabolic/stress-resistance phenotype],0,1,Amivantamab + Lazertinib,0.92,0.24
1,TCGA-91-6847,0.321429,FEMALE,I,LIVING,Uncommon sensitizing EGFR mutation,Other EGFR mutation,"[TP53, KRAS, MET, ERBB2, BRAF, RB1, NRAS, AKT1]","[MET bypass signaling, PI3K/AKT pathway bypass...",1,1,Osimertinib,0.52,0.53
2,TCGA-44-2661,0.571429,FEMALE,I,LIVING,EGFR exon 19 deletion,EGFR exon 19 deletion,"[TP53, MET]","[MET bypass signaling, Genomic instability / a...",1,1,Amivantamab + Lazertinib,0.89,0.32
3,TCGA-05-4402,0.142857,FEMALE,IV,DECEASED,EGFR exon 19 deletion,EGFR exon 19 deletion,"[TP53, KRAS, PIK3CA, MET, ERBB2, BRAF, NF1, NR...","[MET bypass signaling, PI3K/AKT pathway bypass...",1,1,Amivantamab + Lazertinib,0.89,0.32
4,TCGA-64-1681,0.285714,FEMALE,I,DECEASED,EGFR L858R,EGFR L858R,"[TP53, KRAS, MET, BRAF, NRAS]","[MET bypass signaling, MAPK pathway bypass, Ge...",1,1,Amivantamab + Lazertinib,0.89,0.32
5,TCGA-69-7765,0.107143,MALE,IV,LIVING,Rare/uncertain EGFR mutation,Other EGFR mutation,"[TP53, MET, BRAF, NF1, AKT1]","[MET bypass signaling, PI3K/AKT pathway bypass...",1,1,Osimertinib,0.52,0.53
6,TCGA-50-6595,0.750000,FEMALE,III,DECEASED,Uncommon sensitizing EGFR mutation,Other EGFR mutation,"[TP53, PIK3CA]","[PI3K/AKT pathway bypass, Genomic instability ...",1,1,Osimertinib,0.52,0.53
7,TCGA-49-4494,0.857143,MALE,III,DECEASED,Compound EGFR mutation with T790M resistance,EGFR T790M resistance,"[MET, BRAF]","[EGFR T790M-mediated resistance, MET bypass si...",0,1,Osimertinib,0.82,0.42
8,TCGA-38-6178,0.607143,FEMALE,III,LIVING,EGFR exon 19 deletion,EGFR exon 19 deletion,"[TP53, MET, BRAF, NRAS]","[MET bypass signaling, MAPK pathway bypass, Ge...",1,1,Amivantamab + Lazertinib,0.89,0.32
9,TCGA-38-4627,0.392857,FEMALE,II,DECEASED,EGFR L858R,EGFR L858R,"[MET, BRAF, AKT1]","[MET bypass signaling, PI3K/AKT pathway bypass...",0,1,Amivantamab + Lazertinib,0.92,0.24



Unknown EGFR count:
0 / 23

Unknown sex count:
0 / 23

Unknown stage count:
0 / 23


In [ ]:
from pathlib import Path
import json

pred_path = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json")

print("Exists:", pred_path.exists())
print("Path:", pred_path)

if pred_path.exists():
    with open(pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)

    print("Number of predictions:", len(preds))
    print("First patient:")
    print(preds[0])
else:
    print(f"\nLe fichier n'existe pas : {pred_path}")
    print("Veuillez vérifier le chemin ou vous assurer que le fichier a bien été généré.")
    preds = []


Exists: True
Path: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json
Number of predictions: 23
First patient:
{'patient_id': 'TCGA-67-6217', 'clinical_profile': {'age': 0.7142857142857144, 'sex': 'FEMALE', 'stage': 'II', 'survival_status': 'LIVING'}, 'molecular_profile': {'egfr_raw': 'EGFR L858R', 'egfr_class': 'EGFR L858R', 'secondary_mutations': ['STK11', 'KEAP1', 'NRAS'], 'resistance_mechanisms': ['Metabolic/stress-resistance phenotype'], 'has_TP53': 0, 'has_MET': 1, 'has_KRAS': 0, 'has_PIK3CA': 0}, 'pathological_profile': {'pathology_score': 0.5, 'tumor_aggressiveness': 0.3911927959598762, 'immune_score': 0.5, 'egfr_pathway_activity': 0.3515361616932053}, 'best_prediction': {'drug': 'Amivantamab + Lazertinib', 'predicted_efficiency': 0.917, 'resistance_risk': 0.254, 'best_treatment_probability': 0.813, 'global_score': 0.774, 'tunisia_feasibility': 'Très limité / importation ou recherche', 'tunisia_feasibility_score': 0.2, 'local_alter

In [ ]:
from pathlib import Path
import json, shutil, os

BASE_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP")
EXPORT_DIR = BASE_DIR / "exports"
APP_DIR = BASE_DIR / "desktop_app"
APP_DATA_DIR = APP_DIR / "data"
APP_MODELS_DIR = APP_DIR / "models"

APP_DATA_DIR.mkdir(parents=True, exist_ok=True)
APP_MODELS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

expected_path = APP_DATA_DIR / "egfr_ai_predictions_all_patients.json"

print("Expected app file:")
print(expected_path)
print("Exists:", expected_path.exists())

# 1) Search existing prediction files
candidates = list(BASE_DIR.rglob("*predictions*patients*.json"))

print("\nPrediction candidates found:")
for i, p in enumerate(candidates):
    print(i, p)

# 2) If a prediction file exists, copy it to the expected app path
if candidates:
    # Prefer fixed file if available
    fixed_candidates = [p for p in candidates if "fixed" in p.name.lower()]
    source = fixed_candidates[0] if fixed_candidates else candidates[0]

    shutil.copy2(source, expected_path)

    print("\nCopied:")
    print("FROM:", source)
    print("TO:", expected_path)

# 3) If no file exists but all_predictions exists in memory, save it
elif "all_predictions" in globals():
    with open(expected_path, "w", encoding="utf-8") as f:
        json.dump(all_predictions, f, indent=2, ensure_ascii=False)

    with open(EXPORT_DIR / "egfr_ai_predictions_all_patients_fixed.json", "w", encoding="utf-8") as f:
        json.dump(all_predictions, f, indent=2, ensure_ascii=False)

    print("\nSaved all_predictions from memory to:")
    print(expected_path)

# 4) If neither file nor variable exists
else:
    print("\n⚠️ ATTENTION: No prediction JSON found and all_predictions variable does not exist.")
    print("You need to run the cells that generate AI predictions before launching the API.")

# Final check
print("\nFinal check:")
print("Exists:", expected_path.exists())

if expected_path.exists():
    with open(expected_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    print("Number of prediction records:", len(preds))
    print("First patient ID:", preds[0].get("patient_id"))
    print("First best treatment:", preds[0].get("best_prediction", {}).get("drug"))
else:
    print("Impossible de charger les prédictions car le fichier est introuvable.")


Expected app file:
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json
Exists: True

Prediction candidates found:
0 /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/exports/egfr_ai_predictions_all_patients_fixed.json
1 /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json

Copied:
FROM: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/exports/egfr_ai_predictions_all_patients_fixed.json
TO: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json

Final check:
Exists: True
Number of prediction records: 23
First patient ID: TCGA-67-6217
First best treatment: Amivantamab + Lazertinib


In [ ]:
# ============================================================
# RESCUE COMPLET : TRAIN MODELS + GENERATE PREDICTION JSON
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json, shutil, os, ast
import joblib

BASE_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
EXPORT_DIR = BASE_DIR / "exports"
APP_DIR = BASE_DIR / "desktop_app"
APP_DATA_DIR = APP_DIR / "data"
APP_MODELS_DIR = APP_DIR / "models"

for d in [DATA_DIR, MODEL_DIR, EXPORT_DIR, APP_DIR, APP_DATA_DIR, APP_MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

patients_path = DATA_DIR / "patients_model_table_fixed.csv"
therapy_path = DATA_DIR / "patient_treatment_training_table_fixed.csv"

if not patients_path.exists() or not therapy_path.exists():
    raise Exception("""
Les fichiers fixed sont introuvables.
Tu dois d'abord relancer la cellule d'extraction corrigée qui crée :
- patients_model_table_fixed.csv
- patient_treatment_training_table_fixed.csv
""")

patients_df = pd.read_csv(patients_path)
therapy_df = pd.read_csv(therapy_path)

print("Patients loaded:", patients_df.shape)
print("Therapies loaded:", therapy_df.shape)

# -----------------------------
# Helpers
# -----------------------------
def parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, list):
                return v
        except:
            pass
        if x.strip() == "":
            return []
        return [x]
    return [str(x)]

def normalize_score(x, default=0.5):
    try:
        x = float(x)
        if x > 1:
            x = x / 100
        return float(np.clip(x, 0, 1))
    except:
        return default

def tunisia_availability_status(drug):
    d = str(drug).lower()

    if "amivantamab" in d or "lazertinib" in d:
        return "Très limité / importation ou recherche"
    if "osimertinib" in d:
        return "Limité / vérifier pharmacie hospitalière"
    if "afatinib" in d or "gefitinib" in d or "erlotinib" in d or "dacomitinib" in d:
        return "Alternative possible / vérifier disponibilité"
    if "platinum" in d or "chemotherapy" in d or "chimiothérapie" in d:
        return "Option locale plus accessible"

    return "Disponibilité inconnue / vérifier"

def tunisia_feasibility_score(drug):
    status = tunisia_availability_status(drug)

    if "plus accessible" in status:
        return 0.85
    if "Alternative possible" in status:
        return 0.65
    if "Limité" in status:
        return 0.40
    if "Très limité" in status:
        return 0.20
    return 0.50

def local_alternative(drug, egfr_class):
    d = str(drug).lower()

    if "amivantamab" in d or "lazertinib" in d:
        return "Osimertinib si disponible ; sinon erlotinib/géfitinib/afatinib ou chimiothérapie à base de platine"
    if "osimertinib" in d:
        return "Erlotinib/géfitinib/afatinib ou chimiothérapie à base de platine selon disponibilité"
    if "immunotherapy" in d:
        return "Chimiothérapie à base de platine si immunothérapie indisponible"
    if str(egfr_class) == "EGFR negative":
        return "Protocole local CBNPC selon stade, PD-L1 et état général"

    return "Vérification du protocole local nécessaire"

def drug_design_flag(drug, egfr_class, has_met, has_resistance):
    d = str(drug).lower()

    if "amivantamab" in d or "lazertinib" in d:
        return True
    if str(egfr_class) in ["EGFR T790M resistance", "EGFR exon 20 insertion"]:
        return True
    if int(has_met) == 1 or int(has_resistance) == 1:
        return True
    return False

def tunisian_decision(score):
    score = float(score)
    if score >= 0.72:
        return "Priorité élevée pour traitement de précision"
    if score >= 0.50:
        return "Bénéfice intermédiaire / vérifier disponibilité"
    return "Bénéfice limité / discuter protocole local"

# -----------------------------
# Prepare therapy table
# -----------------------------
required_cols = [
    "patient_id", "age", "sex", "stage", "survival_status", "egfr_class",
    "has_TP53", "has_MET", "has_KRAS", "has_PIK3CA",
    "has_bypass", "has_resistance_mechanism",
    "egfr_pathway_activity", "tumor_aggressiveness", "pathology_score",
    "immune_score", "drug", "target_efficiency", "target_resistance",
    "is_best_treatment"
]

for col in required_cols:
    if col not in therapy_df.columns:
        if col in ["target_efficiency"]:
            therapy_df[col] = 0.55
        elif col in ["target_resistance"]:
            therapy_df[col] = 0.45
        elif col in ["is_best_treatment"]:
            therapy_df[col] = 0
        elif col.startswith("has_"):
            therapy_df[col] = 0
        else:
            therapy_df[col] = "Unknown"

therapy_df["tunisia_availability"] = therapy_df["drug"].apply(tunisia_availability_status)
therapy_df["tunisia_feasibility_score"] = therapy_df["drug"].apply(tunisia_feasibility_score)
therapy_df["local_alternative"] = therapy_df.apply(
    lambda r: local_alternative(r["drug"], r["egfr_class"]),
    axis=1
)
therapy_df["drug_design_research_flag"] = therapy_df.apply(
    lambda r: drug_design_flag(
        r["drug"],
        r["egfr_class"],
        r["has_MET"],
        r["has_resistance_mechanism"]
    ),
    axis=1
)

therapy_df["target_efficiency"] = therapy_df["target_efficiency"].apply(lambda x: normalize_score(x, 0.55))
therapy_df["target_resistance"] = therapy_df["target_resistance"].apply(lambda x: normalize_score(x, 0.45))

therapy_df["tunisian_output_score"] = (
    0.60 * therapy_df["target_efficiency"] +
    0.25 * therapy_df["tunisia_feasibility_score"] -
    0.15 * therapy_df["target_resistance"]
).clip(0, 1)

therapy_df["tunisian_decision"] = therapy_df["tunisian_output_score"].apply(tunisian_decision)

# -----------------------------
# Train models
# -----------------------------
!pip -q install scikit-learn joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor

numeric_cols = [
    "age",
    "has_TP53",
    "has_MET",
    "has_KRAS",
    "has_PIK3CA",
    "has_bypass",
    "has_resistance_mechanism",
    "egfr_pathway_activity",
    "tumor_aggressiveness",
    "pathology_score",
    "immune_score",
    "tunisia_feasibility_score"
]

categorical_cols = [
    "sex",
    "stage",
    "survival_status",
    "egfr_class",
    "drug"
]

for col in numeric_cols:
    if col not in therapy_df.columns:
        therapy_df[col] = 0
    therapy_df[col] = pd.to_numeric(therapy_df[col], errors="coerce").fillna(0)

for col in categorical_cols:
    if col not in therapy_df.columns:
        therapy_df[col] = "Unknown"
    therapy_df[col] = therapy_df[col].fillna("Unknown").astype(str)

feature_cols = categorical_cols + numeric_cols

X = therapy_df[feature_cols]
y_eff = therapy_df["target_efficiency"]
y_res = therapy_df["target_resistance"]
y_best = therapy_df["is_best_treatment"].astype(int)
y_tn = therapy_df["tunisian_output_score"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", "passthrough", numeric_cols)
])

efficiency_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42))
])

resistance_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(n_estimators=300, random_state=43))
])

best_treatment_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=300, random_state=44, class_weight="balanced"))
])

tunisia_output_model = Pipeline([
    ("preprocess", preprocess),
    ("model", GradientBoostingRegressor(random_state=45))
])

efficiency_model.fit(X, y_eff)
resistance_model.fit(X, y_res)
best_treatment_model.fit(X, y_best)
tunisia_output_model.fit(X, y_tn)

joblib.dump(efficiency_model, MODEL_DIR / "efficiency_model.pkl")
joblib.dump(resistance_model, MODEL_DIR / "resistance_model.pkl")
joblib.dump(best_treatment_model, MODEL_DIR / "best_treatment_model.pkl")
joblib.dump(tunisia_output_model, MODEL_DIR / "tunisia_output_model.pkl")

metadata = {
    "model_version": "EGFR_TCGA_Tunisia_AI_v0.1",
    "patients": int(therapy_df["patient_id"].nunique()),
    "training_rows": int(len(therapy_df)),
    "features": feature_cols,
    "treatments": sorted(therapy_df["drug"].dropna().unique().tolist()),
    "note": "Prototype IA entraîné sur données TCGA enrichies avec couche d'adaptation tunisienne."
}

with open(MODEL_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Models trained and saved.")

# -----------------------------
# Simulation engine
# -----------------------------
def simulate_drug_effect(patient, treatment, efficiency, resistance):
    age = normalize_score(patient.get("age", 60), 60)
    stage = str(patient.get("stage", "Unknown"))

    has_tp53 = int(patient.get("has_TP53", 0))
    has_met = int(patient.get("has_MET", 0))
    has_bypass = int(patient.get("has_bypass", 0))
    has_resistance = int(patient.get("has_resistance_mechanism", 0))

    tumor_aggr = normalize_score(patient.get("tumor_aggressiveness", 0.5), 0.5)
    pathology_score = normalize_score(patient.get("pathology_score", 0.5), 0.5)
    immune_score = normalize_score(patient.get("immune_score", 0.5), 0.5)
    egfr_activity = normalize_score(patient.get("egfr_pathway_activity", 0.7), 0.7)

    stage_penalty = {
        "I": 0.04,
        "II": 0.08,
        "III": 0.13,
        "III/IV": 0.16,
        "IV": 0.20,
        "Unknown": 0.12
    }.get(stage, 0.12)

    mutation_penalty = 0
    if has_tp53:
        mutation_penalty += 0.06
    if has_met:
        mutation_penalty += 0.08
    if has_bypass:
        mutation_penalty += 0.08
    if has_resistance:
        mutation_penalty += 0.05

    clinical_risk = float(np.clip(
        stage_penalty + mutation_penalty + tumor_aggr * 0.25 + pathology_score * 0.15,
        0, 1
    ))

    volume = 100.0
    viability = 90.0
    molecular_activity = egfr_activity * 100
    resistance_state = resistance * 100

    timeline = []

    for month in range(1, 13):
        active_drug_effect = max(0.05, efficiency * (1 - (resistance_state / 100) * 0.45))

        shrinkage = active_drug_effect * 8.5
        regrowth = clinical_risk * 5.0 + (resistance_state / 100) * 3.5

        volume = float(np.clip(volume - shrinkage + regrowth, 15, 180))
        viability = float(np.clip(viability - active_drug_effect * 6.5 + clinical_risk * 3.2, 5, 100))
        molecular_activity = float(np.clip(molecular_activity - active_drug_effect * 8 + (resistance_state / 100) * 2.5, 5, 100))
        pathology_risk = float(np.clip(pathology_score * 100 + clinical_risk * 30 + (volume - 50) * 0.10, 0, 100))

        resistance_state = float(np.clip(
            resistance_state + 2.2 + has_met * 0.8 + has_tp53 * 0.5 + has_bypass * 0.8,
            0, 98
        ))

        if volume < 45:
            response = "Réponse importante"
        elif volume < 75:
            response = "Réponse partielle"
        elif volume < 110:
            response = "Stabilisation"
        else:
            response = "Progression"

        timeline.append({
            "month": month,
            "treatment": treatment,
            "tumor_volume": round(volume, 2),
            "cell_viability": round(viability, 2),
            "drug_effect": round(active_drug_effect, 3),
            "resistance": round(resistance_state / 100, 3),
            "molecular_activity": round(molecular_activity, 2),
            "pathology_risk": round(pathology_risk, 2),
            "clinical_risk": round(clinical_risk * 100, 2),
            "response": response
        })

    return timeline

# -----------------------------
# Generate predictions for every patient
# -----------------------------
available_treatments = sorted(therapy_df["drug"].dropna().unique().tolist())

# Ensure patients columns
for col in numeric_cols:
    if col not in patients_df.columns:
        patients_df[col] = 0
    patients_df[col] = pd.to_numeric(patients_df[col], errors="coerce").fillna(0)

for col in ["sex", "stage", "survival_status", "egfr_class"]:
    if col not in patients_df.columns:
        patients_df[col] = "Unknown"
    patients_df[col] = patients_df[col].fillna("Unknown").astype(str)

if "secondary_mutations" not in patients_df.columns:
    patients_df["secondary_mutations"] = "[]"

if "resistance_mechanisms" not in patients_df.columns:
    patients_df["resistance_mechanisms"] = "[]"

if "egfr_raw" not in patients_df.columns:
    patients_df["egfr_raw"] = patients_df["egfr_class"]

all_predictions = []

for _, patient in patients_df.iterrows():
    patient_dict = patient.to_dict()

    candidate_rows = []
    for drug in available_treatments:
        row = patient_dict.copy()
        row["drug"] = drug
        row["tunisia_feasibility_score"] = tunisia_feasibility_score(drug)
        candidate_rows.append(row)

    candidates_df = pd.DataFrame(candidate_rows)

    for col in numeric_cols:
        if col not in candidates_df.columns:
            candidates_df[col] = 0
        candidates_df[col] = pd.to_numeric(candidates_df[col], errors="coerce").fillna(0)

    for col in categorical_cols:
        if col not in candidates_df.columns:
            candidates_df[col] = "Unknown"
        candidates_df[col] = candidates_df[col].fillna("Unknown").astype(str)

    X_candidates = candidates_df[feature_cols]

    eff_pred = efficiency_model.predict(X_candidates)
    res_pred = resistance_model.predict(X_candidates)
    tn_pred = tunisia_output_model.predict(X_candidates)

    try:
        best_proba = best_treatment_model.predict_proba(X_candidates)[:, 1]
    except:
        best_proba = eff_pred - res_pred

    ranked = []

    for i, drug in enumerate(candidates_df["drug"].tolist()):
        eff = float(np.clip(eff_pred[i], 0, 1))
        res = float(np.clip(res_pred[i], 0, 1))
        tn_score = float(np.clip(tn_pred[i], 0, 1))
        best_prob = float(np.clip(best_proba[i], 0, 1))

        global_score = float(np.clip(0.65 * eff + 0.25 * best_prob - 0.10 * res, 0, 1))
        final_tunisia_score = float(np.clip(0.55 * eff + 0.25 * tunisia_feasibility_score(drug) - 0.20 * res, 0, 1))

        ranked.append({
            "drug": drug,
            "predicted_efficiency": round(eff, 3),
            "resistance_risk": round(res, 3),
            "best_treatment_probability": round(best_prob, 3),
            "global_score": round(global_score, 3),
            "tunisia_feasibility": tunisia_availability_status(drug),
            "tunisia_feasibility_score": round(tunisia_feasibility_score(drug), 3),
            "local_alternative": local_alternative(drug, patient_dict.get("egfr_class", "Unknown")),
            "tunisian_output_score": round(final_tunisia_score, 3),
            "tunisian_decision": tunisian_decision(final_tunisia_score),
            "drug_design_research_flag": bool(drug_design_flag(
                drug,
                patient_dict.get("egfr_class", "Unknown"),
                patient_dict.get("has_MET", 0),
                patient_dict.get("has_resistance_mechanism", 0)
            ))
        })

    ranked = sorted(ranked, key=lambda x: (-x["global_score"], x["resistance_risk"]))

    for i, r in enumerate(ranked):
        r["rank"] = i + 1

    best = ranked[0]

    simulation = simulate_drug_effect(
        patient_dict,
        best["drug"],
        best["predicted_efficiency"],
        best["resistance_risk"]
    )

    prediction_record = {
        "patient_id": str(patient_dict.get("patient_id")),
        "clinical_profile": {
            "age": None if pd.isna(patient_dict.get("age")) else float(patient_dict.get("age")),
            "sex": str(patient_dict.get("sex")),
            "stage": str(patient_dict.get("stage")),
            "survival_status": str(patient_dict.get("survival_status"))
        },
        "molecular_profile": {
            "egfr_raw": str(patient_dict.get("egfr_raw")),
            "egfr_class": str(patient_dict.get("egfr_class")),
            "secondary_mutations": parse_list(patient_dict.get("secondary_mutations")),
            "resistance_mechanisms": parse_list(patient_dict.get("resistance_mechanisms")),
            "has_TP53": int(patient_dict.get("has_TP53", 0)),
            "has_MET": int(patient_dict.get("has_MET", 0)),
            "has_KRAS": int(patient_dict.get("has_KRAS", 0)),
            "has_PIK3CA": int(patient_dict.get("has_PIK3CA", 0))
        },
        "pathological_profile": {
            "pathology_score": float(patient_dict.get("pathology_score", 0)),
            "tumor_aggressiveness": float(patient_dict.get("tumor_aggressiveness", 0)),
            "immune_score": float(patient_dict.get("immune_score", 0)),
            "egfr_pathway_activity": float(patient_dict.get("egfr_pathway_activity", 0))
        },
        "best_prediction": best,
        "treatment_ranking": ranked,
        "simulation": simulation,
        "bio_scores": {
            "treatment_sensitivity": round(best["predicted_efficiency"] * 100, 1),
            "resistance_score": round(best["resistance_risk"] * 100, 1),
            "tumor_aggressiveness": round(normalize_score(patient_dict.get("tumor_aggressiveness", 0.5)) * 100, 1),
            "pathology_risk": round(normalize_score(patient_dict.get("pathology_score", 0.5)) * 100, 1),
            "immune_score": round(normalize_score(patient_dict.get("immune_score", 0.5)) * 100, 1),
            "tunisian_output_score": round(best["tunisian_output_score"] * 100, 1)
        },
        "model_version": "EGFR_TCGA_Tunisia_AI_v0.1"
    }

    all_predictions.append(prediction_record)

# -----------------------------
# Save prediction JSON
# -----------------------------
export_json = EXPORT_DIR / "egfr_ai_predictions_all_patients_fixed.json"
app_json = APP_DATA_DIR / "egfr_ai_predictions_all_patients.json"

with open(export_json, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, indent=2, ensure_ascii=False)

with open(app_json, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, indent=2, ensure_ascii=False)

# Copy models to app folder
for file in MODEL_DIR.glob("*.pkl"):
    shutil.copy2(file, APP_MODELS_DIR / file.name)

shutil.copy2(MODEL_DIR / "model_metadata.json", APP_MODELS_DIR / "model_metadata.json")

print("DONE.")
print("Prediction records:", len(all_predictions))
print("Export JSON:", export_json)
print("App JSON:", app_json)
print("App JSON exists:", app_json.exists())
print("First patient:", all_predictions[0]["patient_id"])
print("Best treatment:", all_predictions[0]["best_prediction"]["drug"])

Patients loaded: (23, 22)
Therapies loaded: (230, 27)
Models trained and saved.
DONE.
Prediction records: 23
Export JSON: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/exports/egfr_ai_predictions_all_patients_fixed.json
App JSON: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json
App JSON exists: True
First patient: TCGA-67-6217
Best treatment: Amivantamab + Lazertinib


In [ ]:
from pathlib import Path
import json

pred_path = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json")

print("Exists:", pred_path.exists())
print("Path:", pred_path)

with open(pred_path, "r", encoding="utf-8") as f:
    preds = json.load(f)

print("Number of predictions:", len(preds))
preds[0]

Exists: True
Path: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json
Number of predictions: 23


{'patient_id': 'TCGA-67-6217',
 'clinical_profile': {'age': 0.7142857142857144,
  'sex': 'FEMALE',
  'stage': 'II',
  'survival_status': 'LIVING'},
 'molecular_profile': {'egfr_raw': 'EGFR L858R',
  'egfr_class': 'EGFR L858R',
  'secondary_mutations': ['STK11', 'KEAP1', 'NRAS'],
  'resistance_mechanisms': ['Metabolic/stress-resistance phenotype'],
  'has_TP53': 0,
  'has_MET': 1,
  'has_KRAS': 0,
  'has_PIK3CA': 0},
 'pathological_profile': {'pathology_score': 0.5,
  'tumor_aggressiveness': 0.3911927959598762,
  'immune_score': 0.5,
  'egfr_pathway_activity': 0.3515361616932053},
 'best_prediction': {'drug': 'Amivantamab + Lazertinib',
  'predicted_efficiency': 0.917,
  'resistance_risk': 0.254,
  'best_treatment_probability': 0.813,
  'global_score': 0.774,
  'tunisia_feasibility': 'Très limité / importation ou recherche',
  'tunisia_feasibility_score': 0.2,
  'local_alternative': 'Osimertinib si disponible ; sinon erlotinib/géfitinib/afatinib ou chimiothérapie à base de platine',
  '

In [ ]:
from pathlib import Path
import json

pred_path = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json")

print("Exists:", pred_path.exists())
print("Path:", pred_path)

if pred_path.exists():
    with open(pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)

    print("Number of patients:", len(preds))
    print("First patient:", preds[0]["patient_id"])
    print("Best treatment:", preds[0]["best_prediction"]["drug"])
    print("Efficiency:", preds[0]["best_prediction"]["predicted_efficiency"])
    print("Resistance:", preds[0]["best_prediction"]["resistance_risk"])
else:
    print("STOP: prediction JSON still missing.")

Exists: True
Path: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/data/egfr_ai_predictions_all_patients.json
Number of patients: 23
First patient: TCGA-67-6217
Best treatment: Amivantamab + Lazertinib
Efficiency: 0.917
Resistance: 0.254


In [ ]:
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")
APP_DIR.mkdir(parents=True, exist_ok=True)

api_code = r'''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pathlib import Path
import json

APP_DIR = Path(__file__).resolve().parent
DATA_DIR = APP_DIR / "data"

app = FastAPI(
    title="EGFR Digital Twin AI API",
    version="0.1",
    description="API locale pour prédiction IA, simulation thérapeutique et dashboard EGFR"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

pred_path = DATA_DIR / "egfr_ai_predictions_all_patients.json"

if not pred_path.exists():
    raise RuntimeError(f"Prediction file not found: {pred_path}")

with open(pred_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

@app.get("/")
def root():
    return {
        "message": "EGFR Digital Twin AI API online",
        "patients": len(predictions),
        "status": "running"
    }

@app.get("/api/health")
def health():
    return {
        "status": "ok",
        "patients": len(predictions),
        "model": predictions[0].get("model_version", "unknown") if predictions else "unknown"
    }

@app.get("/api/patients")
def get_patients():
    return [
        {
            "patient_id": p.get("patient_id"),
            "age": p.get("clinical_profile", {}).get("age"),
            "sex": p.get("clinical_profile", {}).get("sex"),
            "stage": p.get("clinical_profile", {}).get("stage"),
            "survival_status": p.get("clinical_profile", {}).get("survival_status"),
            "egfr": p.get("molecular_profile", {}).get("egfr_class"),
            "egfr_raw": p.get("molecular_profile", {}).get("egfr_raw"),
            "best_treatment": p.get("best_prediction", {}).get("drug"),
            "predicted_efficiency": p.get("best_prediction", {}).get("predicted_efficiency"),
            "resistance_risk": p.get("best_prediction", {}).get("resistance_risk"),
            "tunisian_decision": p.get("best_prediction", {}).get("tunisian_decision"),
            "tunisia_feasibility": p.get("best_prediction", {}).get("tunisia_feasibility")
        }
        for p in predictions
    ]

@app.get("/api/patient/{patient_id}")
def get_patient(patient_id: str):
    for p in predictions:
        if str(p.get("patient_id")) == str(patient_id):
            return p
    return {"error": "patient not found"}

@app.get("/api/patient/{patient_id}/ranking")
def get_ranking(patient_id: str):
    for p in predictions:
        if str(p.get("patient_id")) == str(patient_id):
            return p.get("treatment_ranking", [])
    return {"error": "patient not found"}

@app.get("/api/patient/{patient_id}/simulation")
def get_simulation(patient_id: str):
    for p in predictions:
        if str(p.get("patient_id")) == str(patient_id):
            return p.get("simulation", [])
    return {"error": "patient not found"}

@app.get("/api/summary")
def summary():
    treatments = sorted(list(set(
        r.get("drug")
        for p in predictions
        for r in p.get("treatment_ranking", [])
        if r.get("drug")
    )))

    egfr_classes = sorted(list(set(
        p.get("molecular_profile", {}).get("egfr_class")
        for p in predictions
        if p.get("molecular_profile", {}).get("egfr_class")
    )))

    return {
        "patients": len(predictions),
        "available_treatments": treatments,
        "egfr_classes": egfr_classes,
        "model_version": predictions[0].get("model_version", "unknown") if predictions else "unknown"
    }
'''

(APP_DIR / "app.py").write_text(api_code, encoding="utf-8")

print("API created:")
print(APP_DIR / "app.py")

API created:
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/app.py


In [ ]:
!pip -q install fastapi uvicorn

import subprocess, time
from google.colab import output
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")

!pkill -f "uvicorn app:app" || true

api_log = open("/content/egfr_desktop_ai_api.log", "w")

api_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8010"],
    cwd=str(APP_DIR),
    stdout=api_log,
    stderr=api_log
)

time.sleep(3)

api_url = output.eval_js("google.colab.kernel.proxyPort(8010)").rstrip("/")
print("AI API running:")
print(api_url)

print("\nOpen these:")
print(api_url + "/api/health")
print(api_url + "/api/patients")
print(api_url + "/api/summary")


^C
AI API running:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev

Open these:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/health
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/patients
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/summary


In [ ]:
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")
FRONTEND_DIR = APP_DIR / "frontend"
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

index_html = r'''
<!DOCTYPE html>
<html lang="fr">
<head>
  <meta charset="UTF-8">
  <title>EGFR Digital Twin Desktop App</title>

  <style>
    :root {
      --bg: #f1f5f9;
      --card: #ffffff;
      --text: #0f172a;
      --muted: #64748b;
      --blue: #2563eb;
      --green: #16a34a;
      --red: #dc2626;
      --orange: #ea580c;
      --purple: #7e22ce;
      --border: #dbe3ef;
      --dark: #0f172a;
    }

    * {
      box-sizing: border-box;
    }

    body {
      margin: 0;
      font-family: Arial, sans-serif;
      background: var(--bg);
      color: var(--text);
    }

    header {
      background: linear-gradient(135deg, #0f172a, #1e3a8a);
      color: white;
      padding: 22px 32px;
    }

    header h1 {
      margin: 0;
      font-size: 28px;
    }

    header p {
      margin: 6px 0 0;
      color: #dbeafe;
    }

    .layout {
      display: grid;
      grid-template-columns: 310px 1fr;
      min-height: calc(100vh - 92px);
    }

    .sidebar {
      background: white;
      border-right: 1px solid var(--border);
      padding: 18px;
      overflow-y: auto;
    }

    .main {
      padding: 22px;
      overflow-y: auto;
    }

    .card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 20px;
      padding: 18px;
      margin-bottom: 18px;
      box-shadow: 0 6px 16px rgba(15, 23, 42, 0.06);
    }

    .card h2, .card h3 {
      margin: 0 0 14px;
      color: var(--dark);
    }

    .search {
      width: 100%;
      padding: 11px;
      border: 1px solid #cbd5e1;
      border-radius: 12px;
      margin-bottom: 12px;
      font-size: 14px;
    }

    .patient-item {
      padding: 12px;
      border: 1px solid var(--border);
      border-radius: 14px;
      margin-bottom: 10px;
      cursor: pointer;
      background: #f8fafc;
    }

    .patient-item:hover,
    .patient-item.active {
      background: #dbeafe;
      border-color: var(--blue);
    }

    .patient-item b {
      display: block;
      margin-bottom: 4px;
    }

    .small {
      font-size: 13px;
      color: var(--muted);
    }

    .grid2 {
      display: grid;
      grid-template-columns: repeat(2, 1fr);
      gap: 18px;
    }

    .grid3 {
      display: grid;
      grid-template-columns: repeat(3, 1fr);
      gap: 18px;
    }

    .stat {
      font-size: 34px;
      font-weight: bold;
      color: var(--blue);
      margin-top: 6px;
    }

    .pill {
      display: inline-block;
      padding: 6px 10px;
      border-radius: 999px;
      font-size: 12px;
      font-weight: bold;
      color: white;
      margin: 3px;
      background: var(--blue);
    }

    .green { background: var(--green); }
    .red { background: var(--red); }
    .orange { background: var(--orange); }
    .purple { background: var(--purple); }
    .gray { background: #64748b; }

    table {
      width: 100%;
      border-collapse: collapse;
      overflow: hidden;
      border-radius: 16px;
    }

    th, td {
      padding: 11px;
      border-bottom: 1px solid #e5e7eb;
      text-align: left;
      font-size: 14px;
    }

    th {
      background: #0f172a;
      color: white;
    }

    canvas {
      width: 100%;
      height: 280px;
      border: 1px solid #e2e8f0;
      border-radius: 16px;
      background: white;
    }

    button {
      border: none;
      padding: 10px 14px;
      border-radius: 12px;
      font-weight: bold;
      cursor: pointer;
      margin: 4px;
    }

    .btn-primary { background: var(--blue); color: white; }
    .btn-secondary { background: #e2e8f0; color: #0f172a; }

    .alert {
      background: #fff7ed;
      border-left: 5px solid var(--orange);
      padding: 14px;
      border-radius: 14px;
      color: #7c2d12;
      line-height: 1.5;
      margin-bottom: 18px;
    }

    pre {
      background: #020617;
      color: #d1fae5;
      padding: 15px;
      border-radius: 16px;
      overflow: auto;
      max-height: 320px;
    }

    @media(max-width: 1000px) {
      .layout {
        grid-template-columns: 1fr;
      }
      .grid2, .grid3 {
        grid-template-columns: 1fr;
      }
    }
  </style>
</head>

<body>

<header>
  <h1>EGFR Lung Cancer Digital Twin</h1>
  <p>Application desktop de prédiction thérapeutique, simulation tumorale et adaptation au contexte tunisien</p>
</header>

<div class="layout">

  <aside class="sidebar">
    <h3>Patients TCGA</h3>
    <input id="searchInput" class="search" placeholder="Rechercher patient, EGFR, stade..." oninput="renderPatients()">
    <div id="patientList"></div>
  </aside>

  <main class="main">

    <div class="alert">
      <b>Prototype de recherche :</b> ce système prédit la réponse thérapeutique à partir de profils TCGA enrichis.
      La couche tunisienne estime la faisabilité locale et ne remplace pas la décision médicale.
    </div>

    <section class="grid3">
      <div class="card">
        <div class="small">Patients chargés</div>
        <div class="stat" id="statPatients">0</div>
      </div>

      <div class="card">
        <div class="small">Traitement recommandé</div>
        <div class="stat" id="statTreatment">--</div>
      </div>

      <div class="card">
        <div class="small">Décision Tunisie</div>
        <div class="stat" id="statTunisia">--</div>
      </div>
    </section>

    <section class="grid2">
      <div class="card">
        <h2>Profil clinique</h2>
        <p><b>Patient :</b> <span id="pid">--</span></p>
        <p><b>Âge normalisé :</b> <span id="age">--</span></p>
        <p><b>Sexe :</b> <span id="sex">--</span></p>
        <p><b>Stade :</b> <span id="stage">--</span></p>
        <p><b>Statut de survie :</b> <span id="survival">--</span></p>
      </div>

      <div class="card">
        <h2>Profil moléculaire EGFR</h2>
        <p><b>Mutation EGFR :</b> <span id="egfr">--</span></p>
        <p><b>Mutation brute :</b> <span id="egfrRaw">--</span></p>
        <p><b>Mutations secondaires :</b></p>
        <div id="mutations"></div>
        <p><b>Mécanismes de résistance :</b></p>
        <div id="resistanceMechs"></div>
      </div>
    </section>

    <section class="grid2">
      <div class="card">
        <h2>Bio-scores</h2>
        <p><b>Sensibilité thérapeutique :</b> <span id="scoreSensitivity">--</span>%</p>
        <p><b>Score résistance :</b> <span id="scoreResistance">--</span>%</p>
        <p><b>Agressivité tumorale :</b> <span id="scoreAggressiveness">--</span>%</p>
        <p><b>Risque pathologique :</b> <span id="scorePathology">--</span>%</p>
        <p><b>Score tunisien :</b> <span id="scoreTunisia">--</span>%</p>
      </div>

      <div class="card">
        <h2>Meilleure prédiction IA</h2>
        <p><b>Traitement :</b> <span id="bestDrug">--</span></p>
        <p><b>Efficacité prédite :</b> <span id="bestEff">--</span></p>
        <p><b>Risque de résistance :</b> <span id="bestRes">--</span></p>
        <p><b>Faisabilité Tunisie :</b> <span id="bestTunisia">--</span></p>
        <p><b>Alternative locale :</b> <span id="bestAlternative">--</span></p>
        <p><b>Drug design :</b> <span id="drugDesign">--</span></p>
      </div>
    </section>

    <section class="card">
      <h2>Classement des traitements</h2>
      <table>
        <thead>
          <tr>
            <th>Rang</th>
            <th>Traitement</th>
            <th>Efficacité</th>
            <th>Résistance</th>
            <th>Score global</th>
            <th>Faisabilité Tunisie</th>
            <th>Décision</th>
          </tr>
        </thead>
        <tbody id="rankingBody"></tbody>
      </table>
    </section>

    <section class="grid2">
      <div class="card">
        <h2>Simulation : volume tumoral</h2>
        <canvas id="tumorCanvas" width="700" height="280"></canvas>
      </div>

      <div class="card">
        <h2>Simulation : résistance</h2>
        <canvas id="resCanvas" width="700" height="280"></canvas>
      </div>
    </section>

    <section class="grid2">
      <div class="card">
        <h2>Simulation : viabilité cellulaire</h2>
        <canvas id="viabilityCanvas" width="700" height="280"></canvas>
      </div>

      <div class="card">
        <h2>Simulation : activité moléculaire EGFR</h2>
        <canvas id="molCanvas" width="700" height="280"></canvas>
      </div>
    </section>

    <section class="card">
      <h2>Rapport patient synthétique</h2>
      <button class="btn-primary" onclick="generateReport()">Générer rapport</button>
      <button class="btn-secondary" onclick="window.print()">Imprimer / PDF</button>
      <pre id="reportBox">Sélectionner un patient.</pre>
    </section>

  </main>
</div>

<script>
let PATIENTS = [];
let CURRENT = null;

async function api(path) {
  const res = await fetch(path);
  if (!res.ok) throw new Error(await res.text());
  return await res.json();
}

async function init() {
  PATIENTS = await api("/api/patients");
  document.getElementById("statPatients").textContent = PATIENTS.length;
  renderPatients();

  if (PATIENTS.length > 0) {
    loadPatient(PATIENTS[0].patient_id);
  }
}

function renderPatients() {
  const q = document.getElementById("searchInput").value.toLowerCase();
  const list = document.getElementById("patientList");

  const filtered = PATIENTS.filter(p => {
    return JSON.stringify(p).toLowerCase().includes(q);
  });

  list.innerHTML = filtered.map(p => `
    <div class="patient-item ${CURRENT && CURRENT.patient_id === p.patient_id ? "active" : ""}"
         onclick="loadPatient('${p.patient_id}')">
      <b>${p.patient_id}</b>
      <div class="small">${p.egfr}</div>
      <div class="small">Stade ${p.stage} — ${p.sex}</div>
    </div>
  `).join("");
}

async function loadPatient(patientId) {
  CURRENT = await api(`/api/patient/${patientId}`);
  renderPatients();
  renderPatient(CURRENT);
}

function pill(text, cls="blue") {
  if (!text || text === "[]" || text.length === 0) return '<span class="pill gray">Aucune</span>';
  return `<span class="pill ${cls}">${text}</span>`;
}

function renderPatient(p) {
  const c = p.clinical_profile || {};
  const m = p.molecular_profile || {};
  const patho = p.pathological_profile || {};
  const best = p.best_prediction || {};
  const bio = p.bio_scores || {};

  document.getElementById("pid").textContent = p.patient_id;
  document.getElementById("age").textContent = c.age;
  document.getElementById("sex").textContent = c.sex;
  document.getElementById("stage").textContent = c.stage;
  document.getElementById("survival").textContent = c.survival_status;

  document.getElementById("egfr").textContent = m.egfr_class;
  document.getElementById("egfrRaw").textContent = m.egfr_raw;

  document.getElementById("mutations").innerHTML =
    (m.secondary_mutations || []).map(x => pill(x, "purple")).join("") || pill("Aucune", "gray");

  document.getElementById("resistanceMechs").innerHTML =
    (m.resistance_mechanisms || []).map(x => pill(x, "orange")).join("") || pill("Aucun", "gray");

  document.getElementById("scoreSensitivity").textContent = bio.treatment_sensitivity ?? "--";
  document.getElementById("scoreResistance").textContent = bio.resistance_score ?? "--";
  document.getElementById("scoreAggressiveness").textContent = bio.tumor_aggressiveness ?? "--";
  document.getElementById("scorePathology").textContent = bio.pathology_risk ?? "--";
  document.getElementById("scoreTunisia").textContent = bio.tunisian_output_score ?? "--";

  document.getElementById("bestDrug").textContent = best.drug || "--";
  document.getElementById("bestEff").textContent = best.predicted_efficiency;
  document.getElementById("bestRes").textContent = best.resistance_risk;
  document.getElementById("bestTunisia").textContent = best.tunisia_feasibility || "--";
  document.getElementById("bestAlternative").textContent = best.local_alternative || "--";
  document.getElementById("drugDesign").textContent = best.drug_design_research_flag ? "Oui — piste de recherche" : "Non prioritaire";

  document.getElementById("statTreatment").textContent = best.drug || "--";
  document.getElementById("statTunisia").textContent = best.tunisian_decision || "--";

  renderRanking(p.treatment_ranking || []);
  renderCharts(p.simulation || []);
  generateReport();
}

function renderRanking(ranking) {
  const body = document.getElementById("rankingBody");

  body.innerHTML = ranking.map(r => `
    <tr>
      <td>${r.rank}</td>
      <td><b>${r.drug}</b></td>
      <td>${Math.round(r.predicted_efficiency * 100)}%</td>
      <td>${Math.round(r.resistance_risk * 100)}%</td>
      <td>${Math.round(r.global_score * 100)}%</td>
      <td>${r.tunisia_feasibility}</td>
      <td>${r.tunisian_decision}</td>
    </tr>
  `).join("");
}

function drawLineChart(canvasId, data, key, label) {
  const canvas = document.getElementById(canvasId);
  const ctx = canvas.getContext("2d");

  ctx.clearRect(0, 0, canvas.width, canvas.height);

  const w = canvas.width;
  const h = canvas.height;
  const pad = 42;

  ctx.strokeStyle = "#cbd5e1";
  ctx.lineWidth = 1;

  ctx.beginPath();
  ctx.moveTo(pad, pad);
  ctx.lineTo(pad, h - pad);
  ctx.lineTo(w - pad, h - pad);
  ctx.stroke();

  ctx.fillStyle = "#334155";
  ctx.font = "13px Arial";
  ctx.fillText(label, pad, 20);

  if (!data || data.length === 0) return;

  const values = data.map(d => Number(d[key]));
  const max = Math.max(...values, 100);
  const min = Math.min(...values, 0);

  function x(i) {
    return pad + i * ((w - 2 * pad) / (data.length - 1));
  }

  function y(v) {
    return h - pad - ((v - min) / (max - min || 1)) * (h - 2 * pad);
  }

  ctx.strokeStyle = "#2563eb";
  ctx.lineWidth = 3;
  ctx.beginPath();

  values.forEach((v, i) => {
    if (i === 0) ctx.moveTo(x(i), y(v));
    else ctx.lineTo(x(i), y(v));
  });

  ctx.stroke();

  ctx.fillStyle = "#2563eb";
  values.forEach((v, i) => {
    ctx.beginPath();
    ctx.arc(x(i), y(v), 4, 0, Math.PI * 2);
    ctx.fill();
  });

  ctx.fillStyle = "#64748b";
  ctx.font = "11px Arial";

  data.forEach((d, i) => {
    ctx.fillText(d.month, x(i) - 3, h - 20);
  });
}

function renderCharts(simulation) {
  drawLineChart("tumorCanvas", simulation, "tumor_volume", "Volume tumoral");
  drawLineChart("resCanvas", simulation.map(x => ({...x, resistance: x.resistance * 100})), "resistance", "Résistance (%)");
  drawLineChart("viabilityCanvas", simulation, "cell_viability", "Viabilité cellulaire");
  drawLineChart("molCanvas", simulation, "molecular_activity", "Activité moléculaire EGFR");
}

function generateReport() {
  if (!CURRENT) return;

  const c = CURRENT.clinical_profile || {};
  const m = CURRENT.molecular_profile || {};
  const best = CURRENT.best_prediction || {};
  const bio = CURRENT.bio_scores || {};

  const report = `
RAPPORT SYNTHÉTIQUE - EGFR DIGITAL TWIN

Patient : ${CURRENT.patient_id}
Sexe : ${c.sex}
Âge normalisé : ${c.age}
Stade : ${c.stage}
Statut de survie : ${c.survival_status}

PROFIL MOLÉCULAIRE
Mutation EGFR : ${m.egfr_class}
Mutation brute : ${m.egfr_raw}
Mutations secondaires : ${(m.secondary_mutations || []).join(", ") || "Aucune"}
Mécanismes de résistance : ${(m.resistance_mechanisms || []).join(", ") || "Aucun"}

PRÉDICTION IA
Traitement recommandé : ${best.drug}
Efficacité prédite : ${Math.round((best.predicted_efficiency || 0) * 100)}%
Risque de résistance : ${Math.round((best.resistance_risk || 0) * 100)}%
Décision tunisienne : ${best.tunisian_decision}
Faisabilité : ${best.tunisia_feasibility}
Alternative locale : ${best.local_alternative}

BIO-SCORES
Sensibilité thérapeutique : ${bio.treatment_sensitivity}%
Résistance : ${bio.resistance_score}%
Agressivité tumorale : ${bio.tumor_aggressiveness}%
Risque pathologique : ${bio.pathology_risk}%
Score tunisien : ${bio.tunisian_output_score}%

NOTE
Ce système est un prototype de recherche et d'aide à la visualisation.
Il ne remplace pas la décision médicale multidisciplinaire.
`;

  document.getElementById("reportBox").textContent = report;
}

init().catch(err => {
  alert("Erreur de chargement: " + err.message);
});
</script>

</body>
</html>
'''

(FRONTEND_DIR / "index.html").write_text(index_html, encoding="utf-8")

print("Frontend created:")
print(FRONTEND_DIR / "index.html")

Frontend created:
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/frontend/index.html


In [ ]:
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")
APP_PATH = APP_DIR / "app.py"

txt = APP_PATH.read_text(encoding="utf-8")

if "from fastapi.responses import FileResponse" not in txt:
    txt = txt.replace(
        "from fastapi import FastAPI",
        "from fastapi import FastAPI\nfrom fastapi.responses import FileResponse"
    )

if "FRONTEND_DIR" not in txt:
    txt = txt.replace(
        'DATA_DIR = APP_DIR / "data"',
        'DATA_DIR = APP_DIR / "data"\nFRONTEND_DIR = APP_DIR / "frontend"'
    )

if '@app.get("/dashboard")' not in txt:
    txt += r'''

@app.get("/dashboard")
def dashboard():
    index_path = FRONTEND_DIR / "index.html"
    if not index_path.exists():
        return {"error": "frontend/index.html not found"}
    return FileResponse(index_path)

@app.get("/app")
def app_page():
    index_path = FRONTEND_DIR / "index.html"
    if not index_path.exists():
        return {"error": "frontend/index.html not found"}
    return FileResponse(index_path)
'''

APP_PATH.write_text(txt, encoding="utf-8")

print("API patched to serve dashboard:")
print(APP_PATH)

API patched to serve dashboard:
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/app.py


In [ ]:
!pip -q install fastapi uvicorn

import subprocess, time
from google.colab import output
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")

!pkill -f "uvicorn app:app" || true

api_log = open("/content/egfr_desktop_ai_api.log", "w")

api_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8010"],
    cwd=str(APP_DIR),
    stdout=api_log,
    stderr=api_log
)

time.sleep(3)

api_url = output.eval_js("google.colab.kernel.proxyPort(8010)").rstrip("/")
print("API running:")
print(api_url)
print("Open dashboard:")
print(api_url + "/dashboard")
print("Patients:")
print(api_url + "/api/patients")


^C
API running:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev
Open dashboard:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/dashboard
Patients:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/patients


In [ ]:
from pathlib import Path
import json, math, statistics
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP")
APP_DIR = BASE_DIR / "desktop_app"
DATA_DIR = APP_DIR / "data"

pred_path = DATA_DIR / "egfr_ai_predictions_all_patients.json"

with open(pred_path, "r", encoding="utf-8") as f:
    preds = json.load(f)

print("Patients loaded:", len(preds))
print("First patient age before:", preds[0]["clinical_profile"]["age"])

Patients loaded: 23
First patient age before: 0.7142857142857144


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, re, os

SEARCH_ROOT = Path("/content/drive/MyDrive")

def normalize_pid(x):
    if pd.isna(x):
        return None
    s = str(x)
    m = re.search(r"TCGA[-_][A-Z0-9]{2}[-_][A-Z0-9]{4}", s, re.I)
    if m:
        return m.group(0).replace("_", "-").upper()
    return s.replace("_", "-").upper()

patient_ids = {p["patient_id"] for p in preds}

age_candidates = []

csv_files = list(SEARCH_ROOT.rglob("*.csv"))

for csv in csv_files:
    try:
        df = pd.read_csv(csv, nrows=5000)
        cols = list(df.columns)
        lower_cols = [c.lower() for c in cols]

        pid_cols = [c for c in cols if any(k in c.lower() for k in ["patient", "case", "submitter", "barcode", "bcr"])]
        age_cols = [c for c in cols if "age" in c.lower()]

        if not pid_cols or not age_cols:
            continue

        for pid_col in pid_cols:
            for age_col in age_cols:
                tmp = df[[pid_col, age_col]].copy()
                tmp["patient_id_norm"] = tmp[pid_col].apply(normalize_pid)
                tmp["age_value"] = pd.to_numeric(tmp[age_col], errors="coerce")

                matched = tmp[tmp["patient_id_norm"].isin(patient_ids) & tmp["age_value"].notna()]
                if len(matched) > 0:
                    age_candidates.append({
                        "file": str(csv),
                        "pid_col": pid_col,
                        "age_col": age_col,
                        "matches": len(matched),
                        "data": matched[["patient_id_norm", "age_value"]].drop_duplicates()
                    })

    except Exception:
        pass

print("Age candidate sources found:", len(age_candidates))

for i, c in enumerate(age_candidates[:10]):
    print(i, "matches:", c["matches"], "| file:", c["file"], "| pid:", c["pid_col"], "| age:", c["age_col"])

# Build age map from best candidate
age_map = {}

if age_candidates:
    best = sorted(age_candidates, key=lambda x: x["matches"], reverse=True)[0]
    print("\nUsing best age source:")
    print(best["file"], "|", best["pid_col"], "|", best["age_col"])

    for _, r in best["data"].iterrows():
        age = float(r["age_value"])

        # If TCGA age is in days, convert to years
        if age > 120:
            age = round(age / 365.25, 1)

        # Keep plausible human age
        if 0 < age <= 120:
            age_map[r["patient_id_norm"]] = age

print("Recovered real ages:", len(age_map))

Age candidate sources found: 8
0 matches: 5000 | file: /content/drive/MyDrive/pfe_digital_twin/DigitalTwin_Master_Final.csv | pid: TCGA_Barcode | age: Age at diagnosis
1 matches: 5000 | file: /content/drive/MyDrive/pfe_digital_twin/DigitalTwin_Master_Final.csv | pid: TCGA_Barcode | age: COL6A1|Collagen_VI-R-V
2 matches: 5000 | file: /content/drive/MyDrive/pfe_digital_twin/Twin_Predictive_Final.csv | pid: TCGA_Barcode | age: Age at diagnosis
3 matches: 5000 | file: /content/drive/MyDrive/pfe_digital_twin/Twin_Predictive_Final.csv | pid: TCGA_Barcode | age: COL6A1|Collagen_VI-R-V
4 matches: 46 | file: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/data/patients_model_table.csv | pid: patient_id | age: age
5 matches: 460 | file: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/data/patient_treatment_training_table.csv | pid: patient_id | age: age
6 matches: 23 | file: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/data/patients_model_table_fixed.csv | pid: patient_id | age: age
7 matches: 230 | file: /

In [ ]:
def clamp(x, a=0, b=1):
    return float(max(a, min(b, x)))

def age_from_normalized(x):
    """
    Fallback only.
    Approximate age from normalized value if real age cannot be recovered.
    """
    try:
        x = float(x)
        if 0 <= x <= 1:
            return round(45 + x * 35, 1)  # approx 45-80 range
        return round(x, 1)
    except:
        return None

def simulate_drug_12_months(patient, treatment):
    clinical = patient.get("clinical_profile", {})
    molecular = patient.get("molecular_profile", {})
    patho = patient.get("pathological_profile", {})

    eff = float(treatment.get("predicted_efficiency", 0.55))
    res = float(treatment.get("resistance_risk", 0.45))

    stage = str(clinical.get("stage", "Unknown"))
    age_years = clinical.get("age_years") or age_from_normalized(clinical.get("age"))
    age_risk = 0.0 if not age_years else clamp((age_years - 50) / 40, 0, 1) * 0.08

    has_tp53 = int(molecular.get("has_TP53", 0))
    has_met = int(molecular.get("has_MET", 0))
    has_kras = int(molecular.get("has_KRAS", 0))
    has_pik3ca = int(molecular.get("has_PIK3CA", 0))

    tumor_aggr = float(patho.get("tumor_aggressiveness", 0.5) or 0.5)
    pathology_score = float(patho.get("pathology_score", 0.5) or 0.5)
    immune_score = float(patho.get("immune_score", 0.5) or 0.5)
    egfr_activity = float(patho.get("egfr_pathway_activity", 0.7) or 0.7)

    if tumor_aggr > 1: tumor_aggr /= 100
    if pathology_score > 1: pathology_score /= 100
    if immune_score > 1: immune_score /= 100
    if egfr_activity > 1: egfr_activity /= 100

    stage_risk = {
        "I": 0.06,
        "II": 0.10,
        "III": 0.17,
        "III/IV": 0.21,
        "IV": 0.26,
        "Unknown": 0.14
    }.get(stage, 0.14)

    mutation_risk = 0
    mutation_risk += 0.07 if has_tp53 else 0
    mutation_risk += 0.09 if has_met else 0
    mutation_risk += 0.06 if has_kras else 0
    mutation_risk += 0.05 if has_pik3ca else 0

    biological_risk = clamp(
        stage_risk + mutation_risk + age_risk + tumor_aggr * 0.20 + pathology_score * 0.12 - immune_score * 0.05
    )

    volume = 100.0
    viability = 90.0
    molecular_activity = egfr_activity * 100
    resistance_state = res * 100

    timeline = []

    for month in range(1, 13):
        adaptive_resistance = resistance_state / 100

        active_effect = max(0.03, eff * (1 - adaptive_resistance * 0.48))
        shrinkage = active_effect * 9.0
        regrowth = biological_risk * 5.2 + adaptive_resistance * 3.8

        volume = float(np.clip(volume - shrinkage + regrowth, 10, 200))
        viability = float(np.clip(viability - active_effect * 6.8 + biological_risk * 3.4, 3, 100))
        molecular_activity = float(np.clip(molecular_activity - active_effect * 8.5 + adaptive_resistance * 3.0, 3, 100))

        pathology_risk = float(np.clip(
            pathology_score * 100 + biological_risk * 35 + (volume - 50) * 0.12,
            0, 100
        ))

        resistance_state = float(np.clip(
            resistance_state + 2.0 + has_met * 0.9 + has_tp53 * 0.6 + has_kras * 0.5,
            0, 98
        ))

        if volume < 45:
            response = "Réponse majeure"
        elif volume < 75:
            response = "Réponse partielle"
        elif volume < 110:
            response = "Maladie stable"
        else:
            response = "Progression probable"

        timeline.append({
            "month": month,
            "treatment": treatment["drug"],
            "tumor_volume": round(volume, 2),
            "cell_viability": round(viability, 2),
            "drug_effect": round(active_effect, 3),
            "resistance": round(resistance_state / 100, 3),
            "molecular_activity": round(molecular_activity, 2),
            "pathology_risk": round(pathology_risk, 2),
            "clinical_risk": round(biological_risk * 100, 2),
            "response": response
        })

    return timeline

def clinical_future_from_simulation(treatment, sim):
    final = sim[-1]
    initial_volume = 100
    final_volume = final["tumor_volume"]
    final_resistance = final["resistance"]

    reduction = (initial_volume - final_volume) / initial_volume

    response_probability = clamp(
        treatment.get("predicted_efficiency", 0.5) * 0.75
        + reduction * 0.35
        - treatment.get("resistance_risk", 0.4) * 0.25
    )

    progression_risk = clamp(
        treatment.get("resistance_risk", 0.4) * 0.55
        + final_resistance * 0.30
        + (final_volume > 100) * 0.20
        - response_probability * 0.20
    )

    if response_probability >= 0.70 and progression_risk < 0.45:
        future = "Réponse attendue favorable sur 12 mois"
    elif response_probability >= 0.50 and progression_risk < 0.65:
        future = "Bénéfice possible avec surveillance rapprochée"
    else:
        future = "Risque de progression ou de résistance à surveiller"

    return {
        "expected_12_month_state": future,
        "response_probability": round(response_probability, 3),
        "progression_risk_12m": round(progression_risk, 3),
        "final_tumor_volume": final_volume,
        "final_resistance": final_resistance,
        "final_response_category": final["response"]
    }

def generate_explanations(patient, treatment, future):
    clinical = patient.get("clinical_profile", {})
    molecular = patient.get("molecular_profile", {})

    egfr = molecular.get("egfr_class", "EGFR inconnu")
    stage = clinical.get("stage", "stade inconnu")
    mutations = []

    if molecular.get("has_TP53"): mutations.append("TP53")
    if molecular.get("has_MET"): mutations.append("MET")
    if molecular.get("has_KRAS"): mutations.append("KRAS")
    if molecular.get("has_PIK3CA"): mutations.append("PIK3CA")

    mut_text = ", ".join(mutations) if mutations else "aucune mutation secondaire majeure détectée"

    doctor = (
        f"Le profil {egfr} oriente vers une stratégie ciblée. "
        f"Le traitement {treatment['drug']} présente une efficacité prédite de "
        f"{round(treatment['predicted_efficiency']*100)}% avec un risque de résistance de "
        f"{round(treatment['resistance_risk']*100)}%. "
        f"Le patient est classé stade {stage}. "
        f"À 12 mois, le modèle estime : {future['expected_12_month_state']} "
        f"avec une probabilité de réponse de {round(future['response_probability']*100)}% "
        f"et un risque de progression de {round(future['progression_risk_12m']*100)}%."
    )

    lab = (
        f"Profil moléculaire : {egfr}. Mutations associées : {mut_text}. "
        f"La présence de voies de résistance comme MET ou TP53 peut augmenter le risque "
        f"d'échappement thérapeutique. Le suivi recommandé doit combiner l'évolution clinique, "
        f"les données moléculaires et les marqueurs pathologiques simulés."
    )

    tunisia = (
        f"Contexte tunisien : {treatment.get('tunisia_feasibility')}. "
        f"Alternative proposée : {treatment.get('local_alternative')}. "
        f"Cette couche est indicative et doit être vérifiée selon la pharmacie hospitalière "
        f"et le protocole local."
    )

    return {
        "doctor_explanation": doctor,
        "lab_explanation": lab,
        "tunisia_explanation": tunisia
    }

enriched = []

for p in preds:
    pid = p["patient_id"]
    clinical = p.get("clinical_profile", {})

    pid_norm = pid.upper()
    real_age = age_map.get(pid_norm)

    if real_age is not None:
        clinical["age_years"] = real_age
        clinical["age_source"] = "real_source_file"
    else:
        clinical["age_years"] = age_from_normalized(clinical.get("age"))
        clinical["age_source"] = "estimated_from_normalized_age"

    p["clinical_profile"] = clinical

    simulations_by_treatment = {}

    for treatment in p.get("treatment_ranking", []):
        sim = simulate_drug_12_months(p, treatment)
        future = clinical_future_from_simulation(treatment, sim)
        explanations = generate_explanations(p, treatment, future)

        simulations_by_treatment[treatment["drug"]] = {
            "treatment": treatment,
            "timeline": sim,
            "clinical_future": future,
            "explanations": explanations
        }

        treatment["clinical_future"] = future
        treatment["explanations"] = explanations

    best_drug = p.get("best_prediction", {}).get("drug")
    if best_drug in simulations_by_treatment:
        p["simulation"] = simulations_by_treatment[best_drug]["timeline"]
        p["clinical_future"] = simulations_by_treatment[best_drug]["clinical_future"]
        p["explanations"] = simulations_by_treatment[best_drug]["explanations"]

    p["simulation_by_treatment"] = simulations_by_treatment
    enriched.append(p)

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(enriched, f, indent=2, ensure_ascii=False)

with open(BASE_DIR / "exports" / "egfr_ai_predictions_all_patients_enriched.json", "w", encoding="utf-8") as f:
    json.dump(enriched, f, indent=2, ensure_ascii=False)

print("Enriched predictions saved.")
print("Patients:", len(enriched))
print("First patient:", enriched[0]["patient_id"])
print("Age years:", enriched[0]["clinical_profile"]["age_years"], enriched[0]["clinical_profile"]["age_source"])
print("Treatments simulated:", len(enriched[0]["simulation_by_treatment"]))
print("Best clinical future:", enriched[0].get("clinical_future"))

Enriched predictions saved.
Patients: 23
First patient: TCGA-67-6217
Age years: 73.0 real_source_file
Treatments simulated: 10
Best clinical future: {'expected_12_month_state': 'Réponse attendue favorable sur 12 mois', 'response_probability': 0.76, 'progression_risk_12m': 0.168, 'final_tumor_volume': 61.27, 'final_resistance': 0.602, 'final_response_category': 'Réponse partielle'}


In [ ]:
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")
APP_PATH = APP_DIR / "app.py"

api_code = r'''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pathlib import Path
import json

APP_DIR = Path(__file__).resolve().parent
DATA_DIR = APP_DIR / "data"
FRONTEND_DIR = APP_DIR / "frontend"

app = FastAPI(
    title="EGFR Digital Twin AI API",
    version="0.2",
    description="API locale pour prédiction IA, simulation multi-traitements et explication clinique"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

pred_path = DATA_DIR / "egfr_ai_predictions_all_patients.json"

if not pred_path.exists():
    raise RuntimeError(f"Prediction file not found: {pred_path}")

with open(pred_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

def find_patient(patient_id: str):
    for p in predictions:
        if str(p.get("patient_id")) == str(patient_id):
            return p
    return None

@app.get("/")
def root():
    return {
        "message": "EGFR Digital Twin AI API online",
        "patients": len(predictions),
        "status": "running"
    }

@app.get("/dashboard")
def dashboard():
    index_path = FRONTEND_DIR / "index.html"
    if not index_path.exists():
        return {"error": "frontend/index.html not found"}
    return FileResponse(index_path)

@app.get("/api/health")
def health():
    return {
        "status": "ok",
        "patients": len(predictions),
        "model": predictions[0].get("model_version", "unknown") if predictions else "unknown"
    }

@app.get("/api/patients")
def get_patients():
    return [
        {
            "patient_id": p.get("patient_id"),
            "age_years": p.get("clinical_profile", {}).get("age_years"),
            "age_source": p.get("clinical_profile", {}).get("age_source"),
            "sex": p.get("clinical_profile", {}).get("sex"),
            "stage": p.get("clinical_profile", {}).get("stage"),
            "survival_status": p.get("clinical_profile", {}).get("survival_status"),
            "egfr": p.get("molecular_profile", {}).get("egfr_class"),
            "egfr_raw": p.get("molecular_profile", {}).get("egfr_raw"),
            "best_treatment": p.get("best_prediction", {}).get("drug"),
            "predicted_efficiency": p.get("best_prediction", {}).get("predicted_efficiency"),
            "resistance_risk": p.get("best_prediction", {}).get("resistance_risk"),
            "response_probability": p.get("clinical_future", {}).get("response_probability"),
            "progression_risk_12m": p.get("clinical_future", {}).get("progression_risk_12m"),
            "expected_12_month_state": p.get("clinical_future", {}).get("expected_12_month_state"),
            "tunisian_decision": p.get("best_prediction", {}).get("tunisian_decision"),
            "tunisia_feasibility": p.get("best_prediction", {}).get("tunisia_feasibility")
        }
        for p in predictions
    ]

@app.get("/api/patient/{patient_id}")
def get_patient(patient_id: str):
    p = find_patient(patient_id)
    if p is None:
        return {"error": "patient not found"}
    return p

@app.get("/api/patient/{patient_id}/ranking")
def get_ranking(patient_id: str):
    p = find_patient(patient_id)
    if p is None:
        return {"error": "patient not found"}
    return p.get("treatment_ranking", [])

@app.get("/api/patient/{patient_id}/simulation")
def get_best_simulation(patient_id: str):
    p = find_patient(patient_id)
    if p is None:
        return {"error": "patient not found"}
    return p.get("simulation", [])

@app.get("/api/patient/{patient_id}/simulations")
def get_all_simulations(patient_id: str):
    p = find_patient(patient_id)
    if p is None:
        return {"error": "patient not found"}
    return p.get("simulation_by_treatment", {})

@app.get("/api/patient/{patient_id}/simulation/{drug_name}")
def get_drug_simulation(patient_id: str, drug_name: str):
    p = find_patient(patient_id)
    if p is None:
        return {"error": "patient not found"}

    sims = p.get("simulation_by_treatment", {})

    for drug, payload in sims.items():
        if drug.lower() == drug_name.lower():
            return payload

    return {"error": "drug simulation not found", "available_drugs": list(sims.keys())}

@app.get("/api/patient/{patient_id}/future")
def get_clinical_future(patient_id: str):
    p = find_patient(patient_id)
    if p is None:
        return {"error": "patient not found"}
    return {
        "clinical_future": p.get("clinical_future", {}),
        "explanations": p.get("explanations", {})
    }

@app.get("/api/summary")
def summary():
    treatments = sorted(list(set(
        r.get("drug")
        for p in predictions
        for r in p.get("treatment_ranking", [])
        if r.get("drug")
    )))

    egfr_classes = sorted(list(set(
        p.get("molecular_profile", {}).get("egfr_class")
        for p in predictions
        if p.get("molecular_profile", {}).get("egfr_class")
    )))

    return {
        "patients": len(predictions),
        "available_treatments": treatments,
        "egfr_classes": egfr_classes,
        "model_version": predictions[0].get("model_version", "unknown") if predictions else "unknown"
    }
'''

APP_PATH.write_text(api_code, encoding="utf-8")

print("API v0.2 written:", APP_PATH)

API v0.2 written: /content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/app.py


In [ ]:
!pip -q install fastapi uvicorn

import subprocess, time
from google.colab import output
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")

!pkill -f "uvicorn app:app" || true

api_log = open("/content/egfr_desktop_ai_api.log", "w")

api_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8010"],
    cwd=str(APP_DIR),
    stdout=api_log,
    stderr=api_log
)

time.sleep(3)

api_url = output.eval_js("google.colab.kernel.proxyPort(8010)")
print("API running:")
print(api_url)
print("Open:")
print(api_url + "api/patients")
print(api_url + "api/patient/TCGA-44-2661/simulations")
print(api_url + "api/patient/TCGA-44-2661/future")
print(api_url + "dashboard")

^C
API running:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev
Open:
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.devapi/patients
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.devapi/patient/TCGA-44-2661/simulations
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.devapi/patient/TCGA-44-2661/future
https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.devdashboard


In [ ]:
!pip -q install fastapi uvicorn

import subprocess, time, requests
from pathlib import Path
from google.colab import output

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")

# Stop old servers
!pkill -f "uvicorn app:app" || true

# Start server
api_log = open("/content/egfr_desktop_ai_api.log", "w")

api_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8010"],
    cwd=str(APP_DIR),
    stdout=api_log,
    stderr=api_log
)

time.sleep(4)

# Local test inside Colab
try:
    r = requests.get("http://127.0.0.1:8010/api/health", timeout=5)
    print("LOCAL API STATUS:", r.status_code)
    print(r.text[:500])
except Exception as e:
    print("LOCAL API ERROR:", e)

# Fresh Colab proxy links
base = output.eval_js("google.colab.kernel.proxyPort(8010)")
if not base.endswith("/"):
    base += "/"

print("\nOPEN THESE FRESH LINKS:")
print("Dashboard:", base + "dashboard")
print("Health:", base + "api/health")
print("Patients:", base + "api/patients")
print("Example simulations:", base + "api/patient/TCGA-44-2661/simulations")

^C
LOCAL API STATUS: 200
{"status":"ok","patients":23,"model":"EGFR_TCGA_Tunisia_AI_v0.1"}

OPEN THESE FRESH LINKS:
Dashboard: https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/dashboard
Health: https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/health
Patients: https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/patients
Example simulations: https://8010-m-s-kkb-usw3b0-9u19dv6y7qeg-b.us-west3-0.prod.colab.dev/api/patient/TCGA-44-2661/simulations


In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8010, path="/dashboard")

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import json
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP")
APP_DIR = BASE_DIR / "desktop_app"
DATA_DIR = APP_DIR / "data"

pred_path = DATA_DIR / "egfr_ai_predictions_all_patients.json"

with open(pred_path, "r", encoding="utf-8") as f:
    patients = json.load(f)

AGE_MIN = 53
AGE_MAX = 81

def denormalize_age(age_value):
    try:
        age_value = float(age_value)
        if 0 <= age_value <= 1:
            return int(round(AGE_MIN + age_value * (AGE_MAX - AGE_MIN)))
        return int(round(age_value))
    except:
        return None

for p in patients:
    clinical = p.get("clinical_profile", {})
    normalized_age = clinical.get("age")

    real_age = clinical.get("age_years")
    if real_age is None or real_age == "" or real_age == "None":
        real_age = denormalize_age(normalized_age)

    clinical["age_normalized"] = normalized_age
    clinical["age"] = real_age
    clinical["age_years"] = real_age
    clinical["age_source"] = "dénormalisé depuis la table TCGA"
    p["clinical_profile"] = clinical

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(patients, f, indent=2, ensure_ascii=False)

with open(BASE_DIR / "exports" / "egfr_ai_predictions_no_normalized_age.json", "w", encoding="utf-8") as f:
    json.dump(patients, f, indent=2, ensure_ascii=False)

print("Âges corrigés.")
for p in patients[:5]:
    print(p["patient_id"], "=>", p["clinical_profile"]["age"], "ans")

Âges corrigés.
TCGA-67-6217 => 73.0 ans
TCGA-91-6847 => 56.2 ans
TCGA-44-2661 => 65.0 ans
TCGA-05-4402 => 50.0 ans
TCGA-64-1681 => 55.0 ans


In [ ]:
from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")
FRONTEND_DIR = APP_DIR / "frontend"
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

html = r'''
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>EGFR Lung Cancer Digital Twin</title>

<style>
:root{
  --bg:#07111f;
  --panel:#0f1b2d;
  --panel2:#111f34;
  --card:#13233a;
  --border:#263a57;
  --text:#f8fafc;
  --muted:#94a3b8;
  --cyan:#22d3ee;
  --blue:#3b82f6;
  --green:#22c55e;
  --orange:#f97316;
  --red:#ef4444;
  --purple:#a855f7;
  --yellow:#facc15;
}

*{box-sizing:border-box}

body{
  margin:0;
  font-family:Arial, sans-serif;
  background:radial-gradient(circle at top left,#0b2a45,#07111f 45%,#020617);
  color:var(--text);
}

.topbar{
  height:82px;
  padding:18px 28px;
  border-bottom:1px solid var(--border);
  background:linear-gradient(90deg,#08111f,#102445,#07111f);
  display:flex;
  align-items:center;
  justify-content:space-between;
}

.logo h1{
  margin:0;
  font-size:27px;
  letter-spacing:0.5px;
}

.logo p{
  margin:5px 0 0;
  color:var(--muted);
  font-size:14px;
}

.badge{
  background:#063247;
  border:1px solid var(--cyan);
  color:#a5f3fc;
  padding:8px 12px;
  border-radius:999px;
  font-size:13px;
  font-weight:bold;
}

.app{
  display:grid;
  grid-template-columns:330px 1fr;
  height:calc(100vh - 82px);
}

.sidebar{
  border-right:1px solid var(--border);
  background:rgba(15,27,45,0.95);
  padding:18px;
  overflow:auto;
}

.sidebar h2{
  margin:0 0 14px;
  font-size:20px;
}

.search{
  width:100%;
  background:#07111f;
  border:1px solid var(--border);
  color:white;
  padding:12px;
  border-radius:12px;
  margin-bottom:14px;
  outline:none;
}

.patient{
  padding:13px;
  margin-bottom:11px;
  border-radius:14px;
  border:1px solid var(--border);
  background:#0b1627;
  cursor:pointer;
  transition:.2s;
}

.patient:hover{
  border-color:var(--cyan);
  transform:translateX(2px);
}

.patient.active{
  background:linear-gradient(135deg,#0e7490,#1d4ed8);
  border-color:#67e8f9;
}

.patient b{
  display:block;
  font-size:15px;
  margin-bottom:4px;
}

.patient span{
  display:block;
  font-size:12px;
  color:#cbd5e1;
  line-height:1.4;
}

.main{
  padding:22px;
  overflow:auto;
}

.grid{
  display:grid;
  gap:18px;
}

.grid-3{
  grid-template-columns:1.2fr 1fr 1.1fr;
}

.grid-2{
  grid-template-columns:1fr 1fr;
}

.card{
  background:linear-gradient(180deg,rgba(19,35,58,.98),rgba(15,27,45,.98));
  border:1px solid var(--border);
  border-radius:20px;
  padding:18px;
  box-shadow:0 18px 40px rgba(0,0,0,.22);
}

.card h2{
  margin:0 0 14px;
  font-size:20px;
}

.card h3{
  margin:0 0 10px;
  font-size:16px;
  color:#dbeafe;
}

.info-row{
  display:flex;
  justify-content:space-between;
  border-bottom:1px solid rgba(148,163,184,.12);
  padding:8px 0;
  gap:12px;
}

.info-row .label{
  color:var(--muted);
}

.info-row .value{
  text-align:right;
  font-weight:bold;
}

.big-number{
  font-size:35px;
  color:var(--cyan);
  font-weight:bold;
}

.small{
  color:var(--muted);
  font-size:13px;
}

.pill{
  display:inline-block;
  padding:6px 10px;
  border-radius:999px;
  margin:3px;
  font-size:12px;
  font-weight:bold;
  background:#334155;
}

.pill.cyan{background:#0e7490}
.pill.green{background:#166534}
.pill.red{background:#991b1b}
.pill.orange{background:#9a3412}
.pill.purple{background:#6b21a8}
.pill.blue{background:#1d4ed8}

.progress{
  height:11px;
  background:#0b1627;
  border-radius:999px;
  overflow:hidden;
  border:1px solid var(--border);
}

.progress div{
  height:100%;
  border-radius:999px;
  background:linear-gradient(90deg,var(--cyan),var(--blue));
}

button, select{
  border:none;
  border-radius:12px;
  padding:11px 14px;
  font-weight:bold;
}

button{
  cursor:pointer;
  color:white;
  background:linear-gradient(135deg,#0891b2,#2563eb);
}

button.secondary{
  background:#1e293b;
  border:1px solid var(--border);
}

select{
  width:100%;
  background:#07111f;
  color:white;
  border:1px solid var(--border);
  margin-bottom:10px;
}

.sim-box{
  display:grid;
  grid-template-columns:260px 1fr;
  gap:18px;
  align-items:start;
}

.month-box{
  background:#07111f;
  border:1px solid var(--border);
  border-radius:18px;
  padding:16px;
}

.month-number{
  font-size:46px;
  color:var(--cyan);
  font-weight:bold;
}

canvas{
  width:100%;
  height:310px;
  background:#07111f;
  border:1px solid var(--border);
  border-radius:18px;
}

table{
  width:100%;
  border-collapse:collapse;
  font-size:13px;
}

th,td{
  padding:10px;
  border-bottom:1px solid rgba(148,163,184,.18);
  text-align:left;
}

th{
  color:#a5f3fc;
  background:#07111f;
}

.explain{
  line-height:1.6;
  color:#e2e8f0;
  background:#07111f;
  border-left:4px solid var(--cyan);
  padding:14px;
  border-radius:14px;
  margin-bottom:12px;
}

.warning{
  border-left-color:var(--orange);
}

@media(max-width:1100px){
  .app{grid-template-columns:1fr}
  .grid-3,.grid-2,.sim-box{grid-template-columns:1fr}
}
</style>
</head>

<body>

<div class="topbar">
  <div class="logo">
    <h1>EGFR Lung Cancer Digital Twin</h1>
    <p>Prédiction IA · Simulation dynamique 12 mois · Adaptation au contexte tunisien</p>
  </div>
  <div class="badge">Prototype PFE — Médecine de précision</div>
</div>

<div class="app">

<aside class="sidebar">
  <h2>Patients TCGA</h2>
  <input class="search" id="search" placeholder="Rechercher patient, mutation, stade..." oninput="renderPatientList()">
  <div id="patientList"></div>
</aside>

<main class="main">

  <div class="grid grid-3">
    <div class="card">
      <h2>Patient sélectionné</h2>
      <div class="big-number" id="selectedPatient">--</div>
      <div class="small" id="selectedSubtitle">--</div>
    </div>

    <div class="card">
      <h2>Traitement IA</h2>
      <div class="big-number" id="bestTreatmentShort">--</div>
      <div class="small">traitement recommandé par le modèle</div>
    </div>

    <div class="card">
      <h2>État prédit à 12 mois</h2>
      <div class="big-number" id="futureStateShort">--</div>
      <div class="small">selon simulation du traitement choisi</div>
    </div>
  </div>

  <br>

  <div class="grid grid-3">
    <section class="card">
      <h2>Résumé clinique</h2>
      <div class="info-row"><span class="label">Âge</span><span class="value" id="age">--</span></div>
      <div class="info-row"><span class="label">Sexe</span><span class="value" id="sex">--</span></div>
      <div class="info-row"><span class="label">Stade tumoral</span><span class="value" id="stage">--</span></div>
      <div class="info-row"><span class="label">Statut de survie</span><span class="value" id="survival">--</span></div>
    </section>

    <section class="card">
      <h2>Profil moléculaire</h2>
      <div class="info-row"><span class="label">EGFR</span><span class="value" id="egfr">--</span></div>
      <div class="info-row"><span class="label">Mutation brute</span><span class="value" id="egfrRaw">--</span></div>
      <h3>Altérations secondaires</h3>
      <div id="mutations"></div>
      <h3>Mécanismes de résistance</h3>
      <div id="resistanceMechs"></div>
    </section>

    <section class="card">
      <h2>Scores biologiques</h2>
      <div class="info-row"><span class="label">Sensibilité</span><span class="value" id="sensScore">--</span></div>
      <div class="progress"><div id="sensBar"></div></div><br>

      <div class="info-row"><span class="label">Résistance</span><span class="value" id="resScore">--</span></div>
      <div class="progress"><div id="resBar"></div></div><br>

      <div class="info-row"><span class="label">Agressivité</span><span class="value" id="aggScore">--</span></div>
      <div class="progress"><div id="aggBar"></div></div><br>

      <div class="info-row"><span class="label">Score tunisien</span><span class="value" id="tnScore">--</span></div>
      <div class="progress"><div id="tnBar"></div></div>
    </section>
  </div>

  <br>

  <section class="card">
    <h2>Simulation dynamique du traitement</h2>

    <div class="sim-box">
      <div class="month-box">
        <label class="small">Choisir un médicament</label>
        <select id="drugSelect"></select>

        <button onclick="startSimulation()">Lancer simulation 12 mois</button>
        <button class="secondary" onclick="resetSimulation()">Réinitialiser</button>

        <br><br>

        <div class="small">Mois simulé</div>
        <div class="month-number" id="monthNumber">0</div>

        <div class="info-row"><span class="label">Réponse</span><span class="value" id="monthResponse">--</span></div>
        <div class="info-row"><span class="label">Volume tumoral</span><span class="value" id="monthVolume">--</span></div>
        <div class="info-row"><span class="label">Viabilité</span><span class="value" id="monthViability">--</span></div>
        <div class="info-row"><span class="label">Effet médicament</span><span class="value" id="monthDrugEffect">--</span></div>
        <div class="info-row"><span class="label">Résistance</span><span class="value" id="monthResistance">--</span></div>
      </div>

      <div>
        <canvas id="mainChart" width="1000" height="310"></canvas>
      </div>
    </div>
  </section>

  <br>

  <div class="grid grid-2">
    <section class="card">
      <h2>Interprétation médicale</h2>
      <div class="explain" id="doctorExplanation">Sélectionner un patient et lancer une simulation.</div>
      <div class="explain warning" id="tunisiaExplanation">La faisabilité tunisienne sera affichée ici.</div>
    </section>

    <section class="card">
      <h2>Interprétation laboratoire / anapath</h2>
      <div class="explain" id="labExplanation">Les marqueurs moléculaires et biologiques seront interprétés ici.</div>
    </section>
  </div>

  <br>

  <section class="card">
    <h2>Classement thérapeutique IA</h2>
    <table>
      <thead>
        <tr>
          <th>Rang</th>
          <th>Traitement</th>
          <th>Efficacité</th>
          <th>Résistance</th>
          <th>Réponse 12 mois</th>
          <th>Progression</th>
          <th>Faisabilité Tunisie</th>
        </tr>
      </thead>
      <tbody id="rankingBody"></tbody>
    </table>
  </section>

</main>
</div>

<script>
let PATIENTS = [];
let CURRENT = null;
let animationTimer = null;

async function api(path){
  const res = await fetch(path);
  if(!res.ok){ throw new Error(await res.text()); }
  return await res.json();
}

async function init(){
  PATIENTS = await api("/api/patients");
  renderPatientList();

  if(PATIENTS.length > 0){
    await loadPatient(PATIENTS[0].patient_id);
  }
}

function renderPatientList(){
  const q = document.getElementById("search").value.toLowerCase();
  const container = document.getElementById("patientList");

  const filtered = PATIENTS.filter(p => JSON.stringify(p).toLowerCase().includes(q));

  container.innerHTML = filtered.map(p => `
    <div class="patient ${CURRENT && CURRENT.patient_id === p.patient_id ? "active" : ""}" onclick="loadPatient('${p.patient_id}')">
      <b>${p.patient_id}</b>
      <span>${p.egfr}</span>
      <span>Âge ${p.age_years || "--"} ans · Stade ${p.stage} · ${p.sex}</span>
      <span>${p.best_treatment}</span>
    </div>
  `).join("");
}

async function loadPatient(pid){
  clearInterval(animationTimer);
  CURRENT = await api(`/api/patient/${pid}`);
  renderPatientList();
  renderPatient(CURRENT);
}

function setText(id,val){
  document.getElementById(id).textContent = val ?? "--";
}

function setBar(id,val){
  const v = Math.max(0, Math.min(100, Number(val || 0)));
  document.getElementById(id).style.width = v + "%";
}

function pill(text, cls){
  return `<span class="pill ${cls || "blue"}">${text}</span>`;
}

function renderPatient(p){
  const c = p.clinical_profile || {};
  const m = p.molecular_profile || {};
  const bio = p.bio_scores || {};
  const best = p.best_prediction || {};
  const future = p.clinical_future || {};

  setText("selectedPatient", p.patient_id);
  setText("selectedSubtitle", `${m.egfr_class || "--"} · Stade ${c.stage || "--"} · ${c.sex || "--"}`);

  setText("bestTreatmentShort", best.drug || "--");
  setText("futureStateShort", future.final_response_category || "--");

  setText("age", c.age_years ? `${c.age_years} ans` : "--");
  setText("sex", c.sex);
  setText("stage", c.stage);
  setText("survival", c.survival_status);

  setText("egfr", m.egfr_class);
  setText("egfrRaw", m.egfr_raw);

  const muts = m.secondary_mutations || [];
  const ress = m.resistance_mechanisms || [];

  document.getElementById("mutations").innerHTML =
    muts.length ? muts.map(x => pill(x,"purple")).join("") : pill("Aucune", "blue");

  document.getElementById("resistanceMechs").innerHTML =
    ress.length ? ress.map(x => pill(x,"orange")).join("") : pill("Aucun", "blue");

  setText("sensScore", `${bio.treatment_sensitivity || 0}%`);
  setText("resScore", `${bio.resistance_score || 0}%`);
  setText("aggScore", `${bio.tumor_aggressiveness || 0}%`);
  setText("tnScore", `${bio.tunisian_output_score || 0}%`);

  setBar("sensBar", bio.treatment_sensitivity);
  setBar("resBar", bio.resistance_score);
  setBar("aggBar", bio.tumor_aggressiveness);
  setBar("tnBar", bio.tunisian_output_score);

  renderDrugSelect(p);
  renderRanking(p.treatment_ranking || []);
  resetSimulation();
}

function renderDrugSelect(p){
  const select = document.getElementById("drugSelect");
  const sims = p.simulation_by_treatment || {};
  const drugs = Object.keys(sims);

  select.innerHTML = drugs.map(d => `<option value="${d}">${d}</option>`).join("");

  const best = p.best_prediction?.drug;
  if(best && drugs.includes(best)){
    select.value = best;
  }

  select.onchange = () => {
    resetSimulation();
    updateExplanationsForDrug(select.value);
  };

  updateExplanationsForDrug(select.value);
}

function selectedPayload(){
  const drug = document.getElementById("drugSelect").value;
  const sims = CURRENT.simulation_by_treatment || {};
  return sims[drug];
}

function resetSimulation(){
  clearInterval(animationTimer);
  setText("monthNumber", "0");
  setText("monthResponse", "--");
  setText("monthVolume", "--");
  setText("monthViability", "--");
  setText("monthDrugEffect", "--");
  setText("monthResistance", "--");

  const payload = selectedPayload();
  if(payload && payload.timeline){
    drawChart(payload.timeline, 0);
    updateExplanationsForDrug(document.getElementById("drugSelect").value);
  }
}

function startSimulation(){
  clearInterval(animationTimer);

  const payload = selectedPayload();
  if(!payload || !payload.timeline) return;

  const timeline = payload.timeline;
  let i = 0;

  updateMonth(timeline[0], 1);
  drawChart(timeline, 1);

  animationTimer = setInterval(() => {
    i++;
    if(i >= timeline.length){
      clearInterval(animationTimer);
      return;
    }

    updateMonth(timeline[i], i + 1);
    drawChart(timeline, i + 1);
  }, 750);
}

function updateMonth(m, visibleCount){
  setText("monthNumber", m.month);
  setText("monthResponse", m.response);
  setText("monthVolume", `${m.tumor_volume}%`);
  setText("monthViability", `${m.cell_viability}%`);
  setText("monthDrugEffect", `${Math.round(m.drug_effect * 100)}%`);
  setText("monthResistance", `${Math.round(m.resistance * 100)}%`);
}

function updateExplanationsForDrug(drug){
  const payload = CURRENT?.simulation_by_treatment?.[drug];
  if(!payload) return;

  const exp = payload.explanations || {};
  setText("doctorExplanation", exp.doctor_explanation || "--");
  setText("labExplanation", exp.lab_explanation || "--");
  setText("tunisiaExplanation", exp.tunisia_explanation || "--");
}

function renderRanking(ranking){
  const body = document.getElementById("rankingBody");
  body.innerHTML = ranking.map(r => {
    const fut = r.clinical_future || {};
    return `
      <tr>
        <td>${r.rank}</td>
        <td><b>${r.drug}</b></td>
        <td>${Math.round((r.predicted_efficiency || 0) * 100)}%</td>
        <td>${Math.round((r.resistance_risk || 0) * 100)}%</td>
        <td>${Math.round((fut.response_probability || 0) * 100)}%</td>
        <td>${Math.round((fut.progression_risk_12m || 0) * 100)}%</td>
        <td>${r.tunisia_feasibility}</td>
      </tr>
    `;
  }).join("");
}

function drawChart(timeline, visibleCount){
  const canvas = document.getElementById("mainChart");
  const ctx = canvas.getContext("2d");

  ctx.clearRect(0,0,canvas.width,canvas.height);

  const w = canvas.width;
  const h = canvas.height;
  const pad = 48;

  ctx.strokeStyle = "#263a57";
  ctx.lineWidth = 1;

  ctx.beginPath();
  ctx.moveTo(pad,pad);
  ctx.lineTo(pad,h-pad);
  ctx.lineTo(w-pad,h-pad);
  ctx.stroke();

  const data = timeline.slice(0, visibleCount || timeline.length);

  const series = [
    {key:"tumor_volume", label:"Volume tumoral", color:"#22d3ee"},
    {key:"cell_viability", label:"Viabilité", color:"#22c55e"},
    {key:"molecular_activity", label:"Activité EGFR", color:"#a855f7"},
    {key:"resistance", label:"Résistance", color:"#ef4444", scale:100}
  ];

  function x(i){
    return pad + i * ((w - 2*pad) / 11);
  }

  function y(v){
    return h - pad - (v / 100) * (h - 2*pad);
  }

  ctx.font = "13px Arial";

  series.forEach((s, idx) => {
    ctx.strokeStyle = s.color;
    ctx.lineWidth = 3;
    ctx.beginPath();

    data.forEach((d,i) => {
      let v = Number(d[s.key] || 0);
      if(s.scale) v = v * s.scale;

      if(i === 0) ctx.moveTo(x(i), y(v));
      else ctx.lineTo(x(i), y(v));
    });

    ctx.stroke();

    ctx.fillStyle = s.color;
    ctx.fillText(s.label, pad + idx*180, 24);

    data.forEach((d,i) => {
      let v = Number(d[s.key] || 0);
      if(s.scale) v = v * s.scale;
      ctx.beginPath();
      ctx.arc(x(i), y(v), 4, 0, Math.PI*2);
      ctx.fill();
    });
  });

  ctx.fillStyle = "#94a3b8";
  ctx.font = "12px Arial";
  for(let i=0;i<12;i++){
    ctx.fillText(String(i+1), x(i)-3, h-20);
  }

  ctx.fillText("Mois", w - pad - 15, h - 18);
}

init().catch(err => {
  alert("Erreur dashboard : " + err.message);
});
</script>

</body>
</html>
'''

(FRONTEND_DIR / "index.html").write_text(html, encoding="utf-8")

print("Nouvelle interface v0.3 créée :")
print(FRONTEND_DIR / "index.html")

Nouvelle interface v0.3 créée :
/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app/frontend/index.html


In [ ]:
!pip -q install fastapi uvicorn

import subprocess, time, requests
from pathlib import Path
from google.colab import output

APP_DIR = Path("/content/drive/MyDrive/EGFR_DESKTOP_AI_APP/desktop_app")

!pkill -f "uvicorn app:app" || true

api_log = open("/content/egfr_desktop_ai_api.log", "w")

api_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8010"],
    cwd=str(APP_DIR),
    stdout=api_log,
    stderr=api_log
)

time.sleep(4)

r = requests.get("http://127.0.0.1:8010/api/health", timeout=5)
print("API:", r.status_code, r.text[:200])

output.serve_kernel_port_as_window(8010, path="/dashboard")

^C
API: 200 {"status":"ok","patients":23,"model":"EGFR_TCGA_Tunisia_AI_v0.1"}
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>